# Bat Trajectory Analysis

Cleaned-up pipeline. Sections, in the order they should generally be run:

1. **Data Compilation** — build master trajectory CSVs from the raw per-trajectory files.  -> In other ipynb
2. **Trajectory Smoothing (Savgol Filter)** — smooth the master files (produces the `_smoothed.csv` files used below).  -> In other ipynb
3. **Target Setup** — load target rectangles and provide an interactive trajectory viewer.
4. **Bifurcation Detection** — core algorithm that finds bifurcation points along a trajectory.
5. **Single Trajectory Analysis** — bifurcations + angle-vs-distance plots for one trajectory.
6. **Multi-Trajectory Analysis** — bifurcations across many trajectories, their center of mass, and where trajectories land on the targets.

*Note: sections 4–6 read from `MASTER_FILES`, which is repointed to the `_smoothed` CSVs partway through the notebook — run Section 2 (Smoothing) before Sections 4–6 if the smoothed files don't exist yet.*

## 3. Target Setup

Loads the target rectangles for a given date (`get_rectangles`, `rectangle_center`), sets up the `MASTER_FILES` lookup, and provides an interactive matplotlib 3D viewer for browsing raw trajectories against the targets.

In [1]:
# ============================================================
# Interactive Bat Trajectory Plotter + Target Rectangles
# ============================================================

%matplotlib qt

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

plt.close('all')

# ============================================================
# USER INPUT
# ============================================================

selected_date = "27.10.22"
selected_bat = "motek"
#selected_trajectories = list(range(2, 50))
selected_trajectories = "all"

# ============================================================
# MASTER FILE LOOKUP
# ============================================================

MASTER_FILES = {
    "30.10.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_30Oct_smoothed.csv",
    "24.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_24Dec_smoothed.csv",
    "26.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_26Dec_smoothed.csv",
    "31.12.24": "../csv_acoustic/masters/master_acoustic_trajectory_data_31Dec_smoothed.csv",
    "03.11.22": "../masters_csv/master_vision_trajectory_data_3Nov22_smoothed.csv",
    "01.10.22": "../masters_csv/master_vision_trajectory_data_1Oct22_smoothed.csv",
    "06.10.22": "../masters_csv/master_vision_trajectory_data_6Oct22_smoothed.csv",
    "27.10.22": "../masters_csv/master_vision_trajectory_data_27Oct22_smoothed.csv"
}

if selected_date not in MASTER_FILES:
    raise ValueError(f"No master file configured for {selected_date}")

# ============================================================
# LOAD TRAJECTORIES
# ============================================================

master_df = pd.read_csv(MASTER_FILES[selected_date])

master_df["trajectory_no"] = pd.to_numeric(
    master_df["trajectory_no"], errors="coerce"
)

# ============================================================
# LOAD RECTANGLE DATA
# ============================================================

rectangles_df = pd.read_excel(
    "../csv_acoustic/masters/rectangles_places.xlsx",
    header=2
)

# ============================================================
# RECTANGLE LOADER
#
# Robust to:
#   1) Dates written inconsistently in the sheet (e.g. "6.10.22"
#      vs. "06.10.22") — matched by normalizing day/month/year
#      to integers instead of comparing raw strings.
#   2) Dates with fewer than 3 targets placed (e.g. 6 Oct only
#      has 2 rectangles) — a rectangle is only included if its
#      corner columns are actually filled in for that date.
# ============================================================

def _normalize_date_key(date_str):
    parts = str(date_str).strip().split(".")
    return tuple(int(p) for p in parts)


def get_rectangles(date_num):

    target_key = _normalize_date_key(date_num)

    matches = rectangles_df["Date Num"].apply(
        lambda v: _normalize_date_key(v) == target_key
    )

    row = rectangles_df.loc[matches]

    if row.empty:
        raise ValueError(f"No rectangle data found for {date_num}")

    row = row.iloc[0]

    # (corner_a, corner_b, corner_d, corner_c) column names for
    # each of the up-to-3 possible target rectangles
    rect_column_specs = [
        ("x1_a", "Var4",  "Var5",
         "x1_b", "Var7",  "Var8",
         "x1_d", "Var13", "Var14",
         "x1_c", "Var10", "Var11"),
        ("x2_a", "Var16", "Var17",
         "x2_b", "Var19", "Var20",
         "x2_d", "Var25", "Var26",
         "x2_c", "Var22", "Var23"),
        ("x3_a", "Var43", "Var44",
         "x3_b", "Var46", "Var47",
         "x3_d", "Var52", "Var53",
         "x3_c", "Var49", "Var50"),
    ]

    rects = []

    for cols in rect_column_specs:
        (xa, ya, za,
         xb, yb, zb,
         xd, yd, zd,
         xc, yc, zc) = cols

        # This target wasn't placed on this date -> skip it
        if pd.isna(row[xa]):
            continue

        rect = np.array([
            [row[xa], row[ya], row[za]],
            [row[xb], row[yb], row[zb]],
            [row[xd], row[yd], row[zd]],
            [row[xc], row[yc], row[zc]],
            [row[xa], row[ya], row[za]],
        ])

        rects.append(rect)

    if len(rects) < 2:
        raise ValueError(
            f"Only {len(rects)} target(s) found for {date_num}; "
            f"bifurcation analysis needs at least 2."
        )

    return rects


def rectangle_center(rect):
    return rect[:-1].mean(axis=0)


# Shared color palette used everywhere a variable number of
# targets needs to be drawn/labeled consistently
RECT_COLORS = ["black", "magenta", "cyan", "orange", "purple"]

# ============================================================
# GET RECTANGLES FOR SELECTED DATE
# ============================================================

rects = get_rectangles(selected_date)

# ============================================================
# FILTER BAT
# ============================================================

available_bats = sorted(master_df["bat_name"].unique())

if selected_bat not in available_bats:
    print("\nAvailable bats:")
    print(available_bats)
    raise ValueError(f"'{selected_bat}' not found in dataset.")

bat_df = master_df[master_df["bat_name"] == selected_bat].copy()

# ============================================================
# FILTER TRAJECTORIES
# ============================================================

if selected_trajectories != "all":
    bat_df = bat_df[bat_df["trajectory_no"].isin(selected_trajectories)]

bat_df = bat_df.dropna(subset=["pos_x", "pos_y", "pos_z"])

# ============================================================
# CRITICAL FIX: Sort by trajectory_no AND the original row
# order to preserve point sequence within each trajectory
# ============================================================

bat_df = bat_df.reset_index(drop=False)   # preserve original CSV row index
bat_df = bat_df.sort_values(
    ["trajectory_no", "index"]            # sort by traj, then original order
).reset_index(drop=True)

# ============================================================
# PLOT
# ============================================================

if bat_df.empty:
    print(
        f"No data found for bat '{selected_bat}' "
        f"with trajectories {selected_trajectories}"
    )

else:

    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection="3d")

    # ========================================================
    # PLOT RECTANGLES (any number of targets)
    # ========================================================

    rect_lines = []
    for i, rect in enumerate(rects):
        color = RECT_COLORS[i % len(RECT_COLORS)]
        line, = ax.plot(
            rect[:, 0], rect[:, 1], rect[:, 2],
            color=color, linewidth=3, label=f"Rectangle {i + 1}"
        )
        rect_lines.append(line)

    # ========================================================
    # TRAJECTORY COLORS
    # ========================================================

    unique_trajs = sorted(bat_df["trajectory_no"].unique())
    num_trajs    = len(unique_trajs)
    cmap         = plt.get_cmap("tab20")

    unique_colors = [
        cmap(i) for i in np.linspace(0, 1, max(1, num_trajs))
    ]

    drawn_lines      = list(rect_lines)
    scatter_artists  = []

    # ========================================================
    # PLOT TRAJECTORIES
    # CRITICAL FIX: iterate over unique_trajs directly and
    # slice with boolean mask — never rely on groupby order
    # ========================================================

    for i, traj in enumerate(unique_trajs):

        # Boolean mask guarantees we only get THIS trajectory's rows
        mask    = bat_df["trajectory_no"] == traj
        traj_df = bat_df.loc[mask]          # already sorted by original index

        color = unique_colors[i]

        # CRITICAL FIX: extract numpy arrays before plotting.
        # Passing a DataFrame column directly can let pandas
        # re-index across trajectories in rare edge cases.
        xs = traj_df["pos_x"].to_numpy()
        ys = traj_df["pos_y"].to_numpy()
        zs = traj_df["pos_z"].to_numpy()

        line, = ax.plot(
            xs, ys, zs,
            label=f"Trajectory {int(traj)}",
            color=color,
            marker="o",
            markersize=2,
            linewidth=1
        )

        drawn_lines.append(line)

        sc_start = ax.scatter(
            xs[0], ys[0], zs[0],
            color="green", marker="*", s=150, zorder=5
        )

        sc_end = ax.scatter(
            xs[-1], ys[-1], zs[-1],
            color="red", marker="*", s=150, zorder=5
        )

        scatter_artists.append((sc_start, sc_end))

    # ========================================================
    # AXIS LABELS
    # ========================================================

    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_zlabel("Z Position")
    ax.set_title(f"{selected_bat} | {selected_date}")

    # ========================================================
    # LEGEND — built entirely from Line2D objects so
    # leg.get_lines() has no hidden scatter proxy artists
    # ========================================================

    proxy_start = Line2D(
        [0], [0], marker="*", color="w",
        markerfacecolor="green", markersize=12,
        label="Start Point", linestyle="None"
    )

    proxy_end = Line2D(
        [0], [0], marker="*", color="w",
        markerfacecolor="red", markersize=12,
        label="End Point", linestyle="None"
    )

    legend_handles = drawn_lines + [proxy_start, proxy_end]

    leg = ax.legend(
        handles=legend_handles,
        labels=[h.get_label() for h in legend_handles],
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        title="Objects",
        ncol=1
    )

    # ========================================================
    # CLICK TO HIDE — label-based mapping, never positional
    # ========================================================

    name_to_line = {line.get_label(): line for line in drawn_lines}

    traj_label_to_scatters = {}
    for i, traj in enumerate(unique_trajs):
        label = f"Trajectory {int(traj)}"
        traj_label_to_scatters[label] = scatter_artists[i]

    lined = {}
    for legline in leg.get_lines():
        label = legline.get_label()
        if label in name_to_line:
            legline.set_picker(True)
            legline.set_pickradius(5)
            lined[legline] = label

    def on_pick(event):

        legline = event.artist
        if legline not in lined:
            return

        label    = lined[legline]
        origline = name_to_line[label]
        vis      = not origline.get_visible()
        origline.set_visible(vis)

        if label in traj_label_to_scatters:
            sc_start, sc_end = traj_label_to_scatters[label]
            sc_start.set_visible(vis)
            sc_end.set_visible(vis)

        legline.set_alpha(1.0 if vis else 0.2)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("pick_event", on_pick)

    # ========================================================
    # VIEW
    # ========================================================

    ax.view_init(elev=20, azim=-45)
    plt.tight_layout()
    plt.show()


## 4. Bifurcation Detection

Core algorithm: given a trajectory and a set of targets, finds the points along the trajectory where the bat's heading commits to a target (a "bifurcation").

In [15]:
import numpy as np

def critical_angle_between_targets(position, target_i, target_j):
    vi = target_i - position
    vj = target_j - position
    vi = vi / (np.linalg.norm(vi) + 1e-12)
    vj = vj / (np.linalg.norm(vj) + 1e-12)
    cos_theta = np.clip(np.dot(vi, vj), -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))

def find_bifurcations(
    xs,
    targets,
    required_hold_steps=15,
    window_size=25,
    min_corr_threshold=0.8,
    plateau_hold_steps=1000,
    next_start=500,          # kept for compatibility, no longer used
    trajectory_fraction=0.85,
    angles_deg=None          # NEW – if provided, used instead of computing from xs
):
    """
    If angles_deg is given (shape N-1 x num_targets), it is used directly;
    otherwise the function computes angles from xs and targets as before.
    """
    xs = np.asarray(xs)
    targets = np.asarray(targets)
    num_targets = targets.shape[0]
    N_full = len(xs)

    if num_targets not in (2, 3):
        raise ValueError("find_bifurcations currently supports only 2 or 3 targets")

    # ---------- velocities, positions, distances ----------
    velocities = np.diff(xs, axis=0)
    vel_norms = np.linalg.norm(velocities, axis=1, keepdims=True)
    vel_units = velocities / (vel_norms + 1e-12)
    positions = xs[:-1]

    step_lengths = np.linalg.norm(velocities, axis=1)
    distance = np.concatenate([[0], np.cumsum(step_lengths)])[:-1]

    # ---------- angles (or use supplied) ----------
    if angles_deg is None:
        angles_deg = np.zeros((len(positions), num_targets))
        for i, target in enumerate(targets):
            dir_to_target = target - positions
            dir_norms = np.linalg.norm(dir_to_target, axis=1, keepdims=True)
            dir_units = dir_to_target / (dir_norms + 1e-12)
            dots = np.sum(vel_units * dir_units, axis=1)
            dots = np.clip(dots, -1, 1)
            angles_deg[:, i] = np.degrees(np.arccos(dots))
    else:
        # use the supplied array (must match positions length)
        if len(angles_deg) != len(positions):
            raise ValueError("Supplied angles_deg must have length N-1")

    # ---------- pairwise angle differences ----------
    diff_labels = []
    diff_curves = []
    for i in range(num_targets):
        for j in range(i + 1, num_targets):
            diff_curves.append(np.abs(angles_deg[:, i] - angles_deg[:, j]))
            diff_labels.append(f"|{i}-{j}|")
    diff_curves = np.column_stack(diff_curves)

    # ---------- rolling Pearson correlation (CENTERED window) ----------
    n_points = len(positions)
    num_pairs = diff_curves.shape[1]
    pair_indices = [(i, j) for i in range(num_targets) for j in range(i + 1, num_targets)]

    pair_corr = np.full((n_points, num_pairs), np.nan)
    effective_window = min(window_size, n_points)
    if effective_window < 2:
        effective_window = 2
    half = effective_window // 2
    valid_start = half
    valid_end = n_points - half - (effective_window % 2 == 0)

    if valid_end >= valid_start:
        for p_idx, (i, j) in enumerate(pair_indices):
            a = angles_deg[:, i]
            b = angles_deg[:, j]
            for t in range(valid_start, valid_end + 1):
                start = t - half
                end = t + half + (effective_window % 2)
                a_win = a[start:end]
                b_win = b[start:end]
                corr = np.corrcoef(a_win, b_win)[0, 1]
                if np.isnan(corr):
                    corr = 0.0
                pair_corr[t, p_idx] = corr

    # ---------- standard bifurcation detection ----------
    if num_targets == 2:
        closest_line_idx = np.argmin(angles_deg, axis=1)
        standard_bif_tagged = []
        current_color = closest_line_idx[0]
        hold_count = 0
        for i in range(1, len(closest_line_idx)):
            if closest_line_idx[i] != current_color:
                hold_count += 1
                if hold_count >= required_hold_steps:
                    bif_idx = i - required_hold_steps + 1
                    standard_bif_tagged.append({
                        "index": bif_idx, "pair": (0, 1), "source": "standard"
                    })
                    current_color = closest_line_idx[i]
                    hold_count = 0
            else:
                hold_count = 0
    else:   # num_targets == 3
        if valid_end < valid_start:
            standard_bif_tagged = []
        else:
            outlier_idx_full = np.full(n_points, -1, dtype=int)
            for t in range(valid_start, valid_end + 1):
                row = pair_corr[t]
                if np.all(np.isnan(row)):
                    continue
                best_pair = np.nanargmax(row)
                outlier_map = [2, 1, 0]          # (0,1)->2, (0,2)->1, (1,2)->0
                outlier_idx_full[t] = outlier_map[best_pair]

            standard_bif_tagged = []
            first_valid = valid_start
            current_outlier = outlier_idx_full[first_valid]
            hold_count = 0
            for i in range(first_valid + 1, valid_end + 1):
                if outlier_idx_full[i] == -1:
                    continue
                if outlier_idx_full[i] != current_outlier:
                    hold_count += 1
                    if hold_count >= required_hold_steps:
                        bif_idx = i - required_hold_steps + 1
                        old_pair_k = outlier_map.index(current_outlier)
                        i_t, j_t = pair_indices[old_pair_k]
                        standard_bif_tagged.append({
                            "index": bif_idx, "pair": (i_t, j_t), "source": "standard"
                        })
                        current_outlier = outlier_idx_full[i]
                        hold_count = 0
                else:
                    hold_count = 0

    # ---------- Plateau‑exit detection (global scan) ----------
    min_corr = np.nanmin(pair_corr, axis=1)
    plateau_bif = None
    start_scan = valid_start
    plateau_start = None
    plateau_len = 0
    for t in range(start_scan, valid_end + 1):
        if min_corr[t] >= min_corr_threshold:
            if plateau_start is None:
                plateau_start = t
            plateau_len += 1
        else:
            if plateau_len >= plateau_hold_steps and plateau_start is not None:
                plateau_bif = t
                break
            else:
                plateau_start = None
                plateau_len = 0

    plateau_tagged = []
    if plateau_bif is not None:
        plateau_tagged.append({
            "index": plateau_bif, "pair": (0, 1), "source": "plateau"
        })

    # ---------- Merge (plateau wins) ----------
    all_tagged = standard_bif_tagged + plateau_tagged
    all_tagged.sort(key=lambda b: b["index"])
    final_bif_tagged = []
    for b in all_tagged:
        if not final_bif_tagged:
            final_bif_tagged.append(b)
            continue
        if b["index"] - final_bif_tagged[-1]["index"] <= required_hold_steps:
            if b["source"] == "plateau" and final_bif_tagged[-1]["source"] != "plateau":
                final_bif_tagged[-1] = b
        else:
            final_bif_tagged.append(b)
    final_bif_tagged = final_bif_tagged[:2]

    max_allowed_index = int(N_full * trajectory_fraction)
    final_bif_tagged = [b for b in final_bif_tagged if b["index"] < max_allowed_index]

    # ---------- Chronological roles ----------
    for i, b in enumerate(final_bif_tagged):
        b["role"] = "first" if i == 0 else "second"

    # ---------- NEW Typology & Pair Assignment ----------
    for b in final_bif_tagged:
        idx = b["index"]
        if num_targets == 3:
            corrs = pair_corr[idx]
            # Safety fallback if we somehow hit a NaN row
            if np.all(np.isnan(corrs)):
                min_pair = b["pair"]
            else:
                # Find the pair driving the correlation drop
                min_pair_idx = np.nanargmin(corrs)
                min_pair = pair_indices[min_pair_idx]
            
            b["typology"] = "edge-pair" if min_pair == (0, 1) else "edge-center"
            b["pair"] = min_pair  # Update the pair so critical angle is calculated accurately
        else:
            b["typology"] = "edge-pair"

    # ---------- Attach critical angle ----------
    for b in final_bif_tagged:
        idx = b["index"]
        i_t, j_t = b["pair"]
        b["critical_angle_deg"] = critical_angle_between_targets(
            positions[idx], targets[i_t], targets[j_t]
        )

    return {
        "positions": positions,
        "angles_deg": angles_deg,
        "diff_curves": diff_curves,
        "diff_labels": diff_labels,
        "bif_indices": [b["index"] for b in final_bif_tagged],
        "bifurcations": final_bif_tagged,
        "distance": distance,
        "pair_corr": pair_corr,
        "x_coords": positions[:, 0]
    }

## 5. Single Trajectory Analysis

Runs bifurcation detection on one trajectory and plots the 3D trajectory (with bifurcation points) alongside the angle-to-target vs. distance-travelled curves.

In [19]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

pio.renderers.default = "browser"

# =====================================================
# 3D GEOMETRY HELPER FUNCTIONS (unchanged)
# =====================================================

def get_hitbox_params(rect, depth=300, side_padding=0):
    p0, p1, p2, p3 = rect[0], rect[1], rect[2], rect[3]
    u = p1 - p0
    v = p3 - p0
    L_u = np.linalg.norm(u) + (side_padding * 2)
    L_v = np.linalg.norm(v) + (side_padding * 2)
    u_hat = u / np.linalg.norm(u)
    v_hat = v / np.linalg.norm(v)
    n_hat = np.cross(u_hat, v_hat)
    center = (p0 + p2) / 2.0
    return {"center": center, "u": u_hat, "L_u": L_u,
            "v": v_hat, "L_v": L_v, "n": n_hat, "L_n": depth}

def is_point_in_hitbox(pt, box):
    v = pt - box["center"]
    if abs(np.dot(v, box["u"])) > box["L_u"] / 2.0: return False
    if abs(np.dot(v, box["v"])) > box["L_v"] / 2.0: return False
    if abs(np.dot(v, box["n"])) > box["L_n"] / 2.0: return False
    return True

RECT_COLORS = ["black", "magenta", "cyan", "orange", "purple"]


def analyze_trajectory_plotly(
    selected_date,
    selected_bat,
    selected_trajectory,
    required_hold_steps=50,
    window_size=25,
    min_corr_threshold=0.8,
    plateau_hold_steps=1000,
    next_start=500,
    hitbox_depth=200,
    hitbox_padding=50,
    extrapolation_step=10,
    max_extrapolation=200,
    time_column="Time",
    num_time_ticks=6,
    trajectory_fraction=0.85,
    angle_smoothing_window=51,
    angle_smoothing_polyorder=3,
    master_file_low=None,
    master_file_high=None,
    start_skip=10
):
    # ------------------------- resolve file paths -------------------------
    if master_file_low is None:
        master_file_low = MASTER_FILES[selected_date]
    if master_file_high is None:
        master_file_high = MASTER_FILES[selected_date]

    # ---- load low‑smoothed trajectory ----
    df_low = pd.read_csv(master_file_low)
    df_low["trajectory_no"] = pd.to_numeric(df_low["trajectory_no"], errors="coerce")
    mask_low = (df_low["bat_name"].astype(str) == selected_bat) & \
               (df_low["trajectory_no"] == selected_trajectory)
    traj_low = df_low[mask_low].dropna(subset=["pos_x", "pos_y", "pos_z"])
    if traj_low.empty:
        raise ValueError("Trajectory not found in low‑smoothed file.")
    xs_low = traj_low[["pos_x", "pos_y", "pos_z"]].values
    xs_low = xs_low[start_skip:]

    # ---- load high‑smoothed trajectory ----
    df_high = pd.read_csv(master_file_high)
    df_high["trajectory_no"] = pd.to_numeric(df_high["trajectory_no"], errors="coerce")
    mask_high = (df_high["bat_name"].astype(str) == selected_bat) & \
                (df_high["trajectory_no"] == selected_trajectory)
    traj_high = df_high[mask_high].dropna(subset=["pos_x", "pos_y", "pos_z"])
    if traj_high.empty:
        raise ValueError("Trajectory not found in high‑smoothed file.")
    xs_high = traj_high[["pos_x", "pos_y", "pos_z"]].values
    xs_high = xs_high[start_skip:]

    # time axis (use high file, same for low if identical)
    if time_column in traj_high.columns:
        time_full = traj_high[time_column].values
    else:
        time_full = np.arange(len(traj_high))

    # ---- targets & hitboxes ----
    rects = get_rectangles(selected_date)
    target_centers = np.array([rectangle_center(rect) for rect in rects])
    hitbox_targets = [
        {"name": f"Rectangle {i+1}", "rect": rect,
         "color": RECT_COLORS[i % len(RECT_COLORS)],
         "box": get_hitbox_params(rect, hitbox_depth, hitbox_padding)}
        for i, rect in enumerate(rects)
    ]

    # ---- landing classification (high trajectory) ----
    hit_target = None; hit_target_idx = None; hit_pt = None; hit_idx = None
    extrapolated_pts = []
    for idx_pt, pt in enumerate(xs_high):
        for idx, ht in enumerate(hitbox_targets):
            if is_point_in_hitbox(pt, ht["box"]):
                hit_target = ht; hit_target_idx = idx; hit_pt = pt.copy(); hit_idx = idx_pt
                break
        if hit_target:
            break
    if not hit_target:
        tail_len = max(2, int(len(xs_high) * 0.1))
        velocity = xs_high[-1] - xs_high[-tail_len]
        speed_val = np.linalg.norm(velocity)
        if speed_val > 0:
            direction = velocity / speed_val
            curr_pt = xs_high[-1].copy()
            for _ in range(max_extrapolation):
                curr_pt = curr_pt + direction * extrapolation_step
                extrapolated_pts.append(curr_pt.copy())
                for idx, ht in enumerate(hitbox_targets):
                    if is_point_in_hitbox(curr_pt, ht["box"]):
                        hit_target = ht; hit_target_idx = idx; hit_pt = curr_pt.copy()
                        break
                if hit_target:
                    break
    landing_point = None
    if hit_target and hit_pt is not None:
        box = hit_target["box"]; c = box["center"]; n = box["n"]
        v = hit_pt - c; dist_to_plane = np.dot(v, n)
        landing_point = hit_pt - (dist_to_plane * n)
    end_pt = hit_pt if hit_pt is not None else (extrapolated_pts[-1] if extrapolated_pts else xs_high[-1])

    # ---- truncation at landing ----
    if hit_idx is not None:
        xs_bif_low  = xs_low[:hit_idx+1]
        xs_bif_high = xs_high[:hit_idx+1]
    else:
        xs_bif_low  = xs_low
        xs_bif_high = xs_high

    do_bifurcation = (len(xs_bif_high) >= 2)

    if do_bifurcation:
        target_desc = [
            f"{ht['name']} (landing pt)" if (hit_target is not None and i == hit_target_idx)
            else f"{ht['name']} (center)" for i, ht in enumerate(hitbox_targets)
        ]
        bifurcation_targets = target_centers.copy()
        if hit_target is not None and landing_point is not None:
            bifurcation_targets[hit_target_idx] = landing_point

        # ---- RAW DETECTION (low‑smoothed) ----
        results_raw = find_bifurcations(
            xs_bif_low, bifurcation_targets,
            required_hold_steps=required_hold_steps,
            window_size=window_size,
            min_corr_threshold=min_corr_threshold,
            plateau_hold_steps=plateau_hold_steps,
            next_start=next_start,
            trajectory_fraction=trajectory_fraction)
        pos_raw       = results_raw["positions"]
        dist_low      = results_raw["distance"]
        angles_low    = results_raw["angles_deg"]
        bif_idx_raw   = results_raw["bif_indices"]
        pair_corr_raw = results_raw.get("pair_corr", np.empty((0,0)))
        diff_labels_raw = results_raw["diff_labels"]
        bif_meta_raw  = results_raw.get("bifurcations", [])

        # ---- HIGH TRAJECTORY ANGLES (manually computed) ----
        vel_high  = np.diff(xs_bif_high, axis=0)
        vn_high   = np.linalg.norm(vel_high, axis=1, keepdims=True)
        vu_high   = vel_high / (vn_high + 1e-12)
        pos_high  = xs_bif_high[:-1]
        step_high = np.linalg.norm(vel_high, axis=1)
        dist_high = np.concatenate([[0], np.cumsum(step_high)])[:-1]

        n_targ = len(hitbox_targets)
        angles_high = np.zeros((len(pos_high), n_targ))
        for i, t in enumerate(target_centers):
            d = t - pos_high
            dn = np.linalg.norm(d, axis=1, keepdims=True)
            du = d / (dn + 1e-12)
            dots = np.sum(vu_high * du, axis=1)
            dots = np.clip(dots, -1, 1)
            angles_high[:, i] = np.degrees(np.arccos(dots))

        # time axis for high
        t_bif_high = time_full[:len(xs_bif_high)]
        t_axis_high = t_bif_high[:-1]

        # ---- speed (high) ----
        dt = np.diff(t_bif_high)
        dt = np.where(dt == 0, np.nan, dt)
        speed = np.linalg.norm(vel_high / dt[:, np.newaxis], axis=1)

        # ---- smooth high angles for derivative computation ----
        data_len = len(angles_high)
        win = min(angle_smoothing_window, data_len - (1 if data_len % 2 == 0 else 0))
        if win <= angle_smoothing_polyorder:
            win = data_len - (1 if data_len % 2 == 0 else 0)
            poly = win - 1 if win > 1 else 0
        else:
            poly = angle_smoothing_polyorder
        if win > 2 and poly > 0:
            smooth_high = savgol_filter(angles_high, window_length=win, polyorder=poly, axis=0)
        else:
            smooth_high = angles_high

        d1_high = np.gradient(smooth_high, t_axis_high, axis=0)
        if win > 2 and poly > 0:
            d1_smooth = savgol_filter(d1_high, window_length=win, polyorder=poly, axis=0)
        else:
            d1_smooth = d1_high
        d2_high = np.gradient(d1_smooth, t_axis_high, axis=0)

        # ---- DERIVATIVE DETECTION (on d1_smooth) ----
        results_der = find_bifurcations(
            xs_bif_high, bifurcation_targets,
            required_hold_steps=required_hold_steps,
            window_size=window_size,
            min_corr_threshold=min_corr_threshold,
            plateau_hold_steps=plateau_hold_steps,
            next_start=next_start,
            trajectory_fraction=trajectory_fraction,
            angles_deg=d1_smooth)
        bif_idx_der   = results_der["bif_indices"]
        pair_corr_der = results_der["pair_corr"]
        min_corr_der  = np.nanmin(pair_corr_der, axis=1) if pair_corr_der.size>0 else np.empty(0)
        bif_meta_der  = results_der.get("bifurcations", [])

        # ---- common plotting colours ----
        pair_colors = ["#1f77b4","#2ca02c","#ff7f0e","black"] if n_targ==3 else ["#1f77b4","black"]

        # correlation labels (raw)
        corr_labels_raw = []
        for lbl in diff_labels_raw:
            parts = lbl.strip('|').split('-')
            a = int(parts[0])+1; b = int(parts[1])+1
            corr_labels_raw.append(f"Corr({a},{b})")
        corr_labels_raw.append("Min Corr(1,2,3)")

        # ---- ticks ----
        tick_dist_low = np.linspace(dist_low.min(), dist_low.max(), num_time_ticks)
        tick_dist_high = np.linspace(dist_high.min(), dist_high.max(), num_time_ticks)
        tick_times_low = np.interp(tick_dist_low, dist_low, t_axis_high)
        tick_times_high = np.interp(tick_dist_high, dist_high, t_axis_high)
        txt_dist_low  = [f"{d:.0f}" for d in tick_dist_low]
        txt_dist_high = [f"{d:.0f}" for d in tick_dist_high]
        txt_time_low  = [f"{t:.2f}" for t in tick_times_low]
        txt_time_high = [f"{t:.2f}" for t in tick_times_high]

        # ---- y-ranges (NaN‑safe fix) ----
        ylim = lambda a: (np.nanmin(a), np.nanmax(a)) if a.size > 0 else (0, 1)

        low_ylim    = ylim(angles_low)
        high_ylim   = ylim(angles_high)
        smooth_ylim = ylim(smooth_high)
        corr_ylim   = ylim(pair_corr_raw) if pair_corr_raw.size > 0 else (0, 1)
        if pair_corr_raw.size > 0:
            mcr = np.nanmin(pair_corr_raw, axis=1) if pair_corr_raw.size > 0 else np.empty(0)
            if mcr.size > 0:
                corr_ylim = (
                    np.nanmin([corr_ylim[0], np.nanmin(mcr)]),
                    np.nanmax([corr_ylim[1], np.nanmax(mcr)])
                )
        der_corr_ylim = ylim(pair_corr_der)
        speed_ylim = ylim(speed) if len(speed) > 0 else (0, 1)
        d1_ylim = ylim(d1_high)
        d2_ylim = ylim(d2_high)

    else:
        # fallback (empty arrays, not shown)
        pass

    # ========================================================
    # FIGURE: 4 rows, 3 cols
    # ========================================================
    fig = make_subplots(
        rows=4, cols=3,
        specs=[
            [{"type": "scene", "rowspan": 4}, {"type": "xy"}, {"type": "xy"}],
            [None,                              {"type": "xy"}, {"type": "xy"}],
            [None,                              {"type": "xy"}, {"type": "xy"}],
            [None,                              {"type": "xy"}, {"type": "xy"}]
        ],
        column_widths=[0.45, 0.27, 0.28],
        row_heights=[0.25, 0.25, 0.25, 0.25],
        horizontal_spacing=0.06,
        vertical_spacing=0.08
    )

    # -------- 3D SCENE (high trajectory) --------
    for ht in hitbox_targets:
        rect = ht["rect"]
        fig.add_trace(go.Scatter3d(x=rect[:,0], y=rect[:,1], z=rect[:,2],
                       mode="lines", line=dict(color=ht["color"], width=4), name=ht["name"]),
                       row=1, col=1)
        box = ht["box"]; c = box["center"]
        u_vec = box["u"] * (box["L_u"]/2); v_vec = box["v"] * (box["L_v"]/2); n_vec = box["n"] * (box["L_n"]/2)
        p0 = c - u_vec - v_vec; p1 = c + u_vec - v_vec; p2 = c + u_vec + v_vec; p3 = c - u_vec + v_vec
        padded_face = np.array([p0,p1,p2,p3])
        bottom_face = padded_face - n_vec; top_face = padded_face + n_vec
        b_loop = np.vstack([bottom_face, bottom_face[0]]); t_loop = np.vstack([top_face, top_face[0]])
        def add_box_edge(loop, col):
            fig.add_trace(go.Scatter3d(x=loop[:,0], y=loop[:,1], z=loop[:,2],
                           mode="lines", line=dict(color=col, width=2, dash="dash"),
                           opacity=0.8, showlegend=False), row=1, col=1)
        add_box_edge(b_loop, ht["color"]); add_box_edge(t_loop, ht["color"])
        for j in range(4):
            edge = np.vstack([bottom_face[j], top_face[j]])
            add_box_edge(edge, ht["color"])

    fig.add_trace(go.Scatter3d(x=target_centers[:,0], y=target_centers[:,1], z=target_centers[:,2],
                   mode="markers", marker=dict(color="red", size=6, line=dict(color="black", width=1)),
                   name="Target Centers"), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=xs_high[:,0], y=xs_high[:,1], z=xs_high[:,2],
                   mode="lines", line=dict(color="black", width=3), name="Trajectory"), row=1, col=1)
    if extrapolated_pts:
        ext_arr = np.array([xs_high[-1]] + extrapolated_pts)
        fig.add_trace(go.Scatter3d(x=ext_arr[:,0], y=ext_arr[:,1], z=ext_arr[:,2],
                       mode="lines", line=dict(color="gray", width=3, dash="dash"),
                       opacity=0.6, name="Extrapolated Path"), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[xs_high[0,0]], y=[xs_high[0,1]], z=[xs_high[0,2]],
                   mode="markers", marker=dict(color="green", size=7, symbol="circle"),
                   name="Start"), row=1, col=1)
    end_color = hit_target["color"] if hit_target else "red"
    end_name = f"Landed: {hit_target['name']}" if hit_target else "End (no hit)"
    fig.add_trace(go.Scatter3d(x=[end_pt[0]], y=[end_pt[1]], z=[end_pt[2]],
                   mode="markers", marker=dict(color=end_color, size=8, symbol="square",
                                               line=dict(color="black", width=1)),
                   name=end_name), row=1, col=1)

    # ---- 3D bifurcation markers (red=raw, black=deriv) ----
    if do_bifurcation:
        if len(bif_idx_raw)>0:
            raw_pts = pos_raw[bif_idx_raw]
            hover_raw = []
            if bif_meta_raw:
                for i,b in enumerate(bif_meta_raw):
                    hover_raw.append(f"Bifurcation {i+1} ({'1st' if b['role']=='first' else '2nd'}, {b['typology']})<br>Critical angle = {b['critical_angle_deg']:.1f}°")
            else:
                hover_raw = [f"Bifurcation {i+1}" for i in range(len(bif_idx_raw))]
            fig.add_trace(go.Scatter3d(x=raw_pts[:,0], y=raw_pts[:,1], z=raw_pts[:,2],
                           mode="markers",
                           marker=dict(color="red", size=10, symbol="diamond",
                                       line=dict(color="darkred", width=1)),
                           text=hover_raw, hoverinfo="text+name",
                           name="Bifurcation (raw)"), row=1, col=1)

        if len(bif_idx_der)>0:
            der_pts = pos_high[bif_idx_der]
            hover_der = []
            if bif_meta_der:
                for i,b in enumerate(bif_meta_der):
                    hover_der.append(f"Bifurcation {i+1} ({'1st' if b['role']=='first' else '2nd'}, {b['typology']})<br>Critical angle = {b['critical_angle_deg']:.1f}°")
            else:
                hover_der = [f"Bifurcation {i+1}" for i in range(len(bif_idx_der))]
                
            fig.add_trace(go.Scatter3d(x=der_pts[:,0], y=der_pts[:,1], z=der_pts[:,2],
                           mode="markers",
                           marker=dict(color="black", size=10, symbol="diamond",
                                       line=dict(color="white", width=1)),
                           text=hover_der, hoverinfo="text+name",
                           name="Bifurcation (deriv)"), row=1, col=1)

    # ========================================================
    # Helpers: draw red (raw) & black (deriv) lines using a given distance array
    # ========================================================
    def add_raw_lines(fig, row, col, x_dist, bif_idx_raw, y_range):
        for idx in bif_idx_raw:
            s = x_dist[idx]
            fig.add_trace(go.Scatter(
                x=[s, s], y=y_range, mode="lines",
                line=dict(color="red", dash="dash", width=1.5),
                opacity=0.5, showlegend=False, hoverinfo="skip"),
                row=row, col=col)

    def add_deriv_lines(fig, row, col, x_dist, bif_idx_der, y_range):
        for idx in bif_idx_der:
            s = x_dist[idx]
            fig.add_trace(go.Scatter(
                x=[s, s], y=y_range, mode="lines",
                line=dict(color="black", dash="dash", width=1.5),
                opacity=0.5, showlegend=False, hoverinfo="skip"),
                row=row, col=col)

    # ========================================================
    # COLUMN 2 (low angles, high angles, smoothed high, speed)
    # ========================================================
    if do_bifurcation:
        # Row1 Col2: low-smoothed angles (axis = dist_low)
        for i,ht in enumerate(hitbox_targets):
            fig.add_trace(go.Scatter(x=dist_low, y=angles_low[:,i], mode="lines",
                           line=dict(width=2, color=ht["color"]), name=ht["name"]), row=1, col=2)
        add_raw_lines(fig, 1, 2, dist_low, bif_idx_raw, low_ylim)
        add_deriv_lines(fig, 1, 2, dist_low, bif_idx_der, low_ylim)

        # Row2 Col2: high-smoothed angles (axis = dist_high)
        for i,ht in enumerate(hitbox_targets):
            fig.add_trace(go.Scatter(x=dist_high, y=angles_high[:,i], mode="lines",
                           line=dict(width=2, color=ht["color"]), name=ht["name"]), row=2, col=2)
        add_raw_lines(fig, 2, 2, dist_high, bif_idx_raw, high_ylim)
        add_deriv_lines(fig, 2, 2, dist_high, bif_idx_der, high_ylim)

        # Row3 Col2: smoothed high (axis = dist_high)
        for i,ht in enumerate(hitbox_targets):
            fig.add_trace(go.Scatter(x=dist_high, y=smooth_high[:,i], mode="lines",
                           line=dict(width=2, color=ht["color"]), name=f"smoothed {ht['name']}"), row=3, col=2)
        add_raw_lines(fig, 3, 2, dist_high, bif_idx_raw, smooth_ylim)
        add_deriv_lines(fig, 3, 2, dist_high, bif_idx_der, smooth_ylim)

        # Row4 Col2: speed (axis = dist_high)
        fig.add_trace(go.Scatter(x=dist_high, y=speed, mode="lines",
                       line=dict(color="darkgreen", width=2), name="Speed"), row=4, col=2)
        add_raw_lines(fig, 4, 2, dist_high, bif_idx_raw, speed_ylim)
        add_deriv_lines(fig, 4, 2, dist_high, bif_idx_der, speed_ylim)

        # ---- axes column 2 ----
        titles_c2 = ["Low‑smoothed angle (deg)", "High‑smoothed angle (deg)", "Smoothed high angle (deg)", "Speed (mm/s)"]
        for row, ttl in enumerate(titles_c2, start=1):
            dist = dist_low if row==1 else dist_high
            tick_vals = tick_dist_low if row==1 else tick_dist_high
            tick_text = txt_dist_low if row==1 else txt_dist_high
            fig.update_xaxes(title="Distance Travelled" if row==4 else None,
                             tickmode="array", tickvals=tick_vals, ticktext=tick_text,
                             showgrid=True, gridcolor="lightgray", zeroline=False,
                             showline=True, linecolor="black", linewidth=1, mirror=True,
                             row=row, col=2)
            fig.update_yaxes(title=ttl, showgrid=True, gridcolor="lightgray", zeroline=False,
                             showline=True, linecolor="black", linewidth=1, mirror=True,
                             row=row, col=2)

    # ========================================================
    # COLUMN 3 (d1, d2, raw corr, deriv corr)
    # ========================================================
    if do_bifurcation:
        # Row1 Col3: 1st derivative (axis = dist_high)
        for i,ht in enumerate(hitbox_targets):
            fig.add_trace(go.Scatter(x=dist_high, y=d1_high[:,i], mode="lines",
                           line=dict(width=2, color=ht["color"]), name=f"dθ/dt {ht['name']}"), row=1, col=3)
        add_raw_lines(fig, 1, 3, dist_high, bif_idx_raw, d1_ylim)
        add_deriv_lines(fig, 1, 3, dist_high, bif_idx_der, d1_ylim)

        # Row2 Col3: 2nd derivative (axis = dist_high)
        for i,ht in enumerate(hitbox_targets):
            fig.add_trace(go.Scatter(x=dist_high, y=d2_high[:,i], mode="lines",
                           line=dict(width=2, color=ht["color"]), name=f"d²θ/dt² {ht['name']}"), row=2, col=3)
        add_raw_lines(fig, 2, 3, dist_high, bif_idx_raw, d2_ylim)
        add_deriv_lines(fig, 2, 3, dist_high, bif_idx_der, d2_ylim)

        # Row3 Col3: raw correlation (axis = dist_low)
        if pair_corr_raw.size>0:
            for p in range(pair_corr_raw.shape[1]):
                fig.add_trace(go.Scatter(x=dist_low, y=pair_corr_raw[:,p], mode="lines",
                               line=dict(width=2, color=pair_colors[p]), name=corr_labels_raw[p]), row=3, col=3)
            mcr = np.nanmin(pair_corr_raw, axis=1) if pair_corr_raw.size>0 else np.empty(0)
            if mcr.size>0:
                fig.add_trace(go.Scatter(x=dist_low, y=mcr, mode="lines",
                               line=dict(width=2, color=pair_colors[-1], dash="dash"),
                               name="Min Corr"), row=3, col=3)
        add_raw_lines(fig, 3, 3, dist_low, bif_idx_raw, corr_ylim)
        add_deriv_lines(fig, 3, 3, dist_low, bif_idx_der, corr_ylim)

        # Row4 Col3: derivative correlation (axis = dist_high)
        if pair_corr_der.size>0:
            der_labels = ["dCorr(1,2)","dCorr(1,3)","dCorr(2,3)"] if n_targ==3 else ["dCorr(1,2)"]
            for p in range(pair_corr_der.shape[1]):
                fig.add_trace(go.Scatter(x=dist_high, y=pair_corr_der[:,p], mode="lines",
                               line=dict(width=2, color=pair_colors[p]), name=der_labels[p]), row=4, col=3)
            if min_corr_der.size>0:
                fig.add_trace(go.Scatter(x=dist_high, y=min_corr_der, mode="lines",
                               line=dict(width=2, color=pair_colors[-1], dash="dash"),
                               name="Min dCorr"), row=4, col=3)
        add_raw_lines(fig, 4, 3, dist_high, bif_idx_raw, der_corr_ylim)
        add_deriv_lines(fig, 4, 3, dist_high, bif_idx_der, der_corr_ylim)

        # ---- axes column 3 ----
        titles_c3 = ["d(Angle)/dt (deg/s)", "d²(Angle)/dt² (deg/s²)", "Correlation (raw)", "dCorr"]
        for row, ttl in enumerate(titles_c3, start=1):
            dist = dist_low if row==3 else dist_high
            tick_vals = tick_dist_low if row==3 else tick_dist_high
            tick_text = txt_dist_low if row==3 else txt_dist_high
            fig.update_xaxes(title="Distance Travelled" if row==4 else None,
                             tickmode="array", tickvals=tick_vals, ticktext=tick_text,
                             showgrid=True, gridcolor="lightgray", zeroline=False,
                             showline=True, linecolor="black", linewidth=1, mirror=True,
                             row=row, col=3)
            fig.update_yaxes(title=ttl, showgrid=True, gridcolor="lightgray", zeroline=False,
                             showline=True, linecolor="black", linewidth=1, mirror=True,
                             row=row, col=3)

    # ========================================================
    # TWIN TIME AXES (all panels except 3D)
    # ========================================================
    if do_bifurcation:
        def add_twin(row, col, dist_vals, time_vals, time_texts, y_base):
            subplot = fig.get_subplot(row=row, col=col)
            x_domain = subplot.xaxis.domain
            x_anchor = subplot.xaxis.anchor
            xaxis_name = subplot.xaxis.plotly_name
            yaxis_name = subplot.yaxis.plotly_name
            existing_xaxes = [k for k in fig.layout.to_plotly_json() if k.startswith("xaxis")]
            next_num = len(existing_xaxes) + 1
            twin_key = f"xaxis{next_num}"
            fig.update_layout(**{
                twin_key: dict(
                    overlaying=xaxis_name.replace("xaxis", "x") if xaxis_name != "xaxis" else "x",
                    side="top",
                    matches=xaxis_name.replace("xaxis", "x") if xaxis_name != "xaxis" else "x",
                    domain=x_domain, anchor=x_anchor,
                    tickmode="array", tickvals=dist_vals, ticktext=time_texts,
                    title="Time (s)" if row==1 else None,
                    showgrid=False)
            })
            twin_trace_key = twin_key.replace("xaxis", "x")
            fig.add_trace(go.Scatter(
                x=[dist_vals[0], dist_vals[-1]], y=[y_base, y_base],
                xaxis=twin_trace_key, yaxis=yaxis_name.replace("yaxis", "y"),
                mode="markers", marker=dict(opacity=0), showlegend=False, hoverinfo="skip"))

        # Column 2 panels
        add_twin(1,2, tick_dist_low, tick_times_low, txt_time_low, low_ylim[0])
        for row, yb in [(2,high_ylim[0]), (3,smooth_ylim[0]), (4,speed_ylim[0])]:
            add_twin(row,2, tick_dist_high, tick_times_high, txt_time_high, yb)

        # Column 3 panels
        for row, yb, dist_vals, time_vals, time_texts in [
            (1,d1_ylim[0], tick_dist_high, tick_times_high, txt_time_high),
            (2,d2_ylim[0], tick_dist_high, tick_times_high, txt_time_high),
            (3,corr_ylim[0], tick_dist_low, tick_times_low, txt_time_low),
            (4,der_corr_ylim[0], tick_dist_high, tick_times_high, txt_time_high)
        ]:
            add_twin(row,3, dist_vals, time_vals, time_texts, yb)

    # ========================================================
    # GLOBAL LAYOUT
    # ========================================================
    fig.update_layout(
        template="simple_white",
        title=dict(text=f"{selected_date} | {selected_bat} | Trajectory {selected_trajectory}",
                   x=0.5, xanchor="center", font=dict(size=20, family="Arial")),
        width=2400, height=1100,
        paper_bgcolor="white", plot_bgcolor="white",
        font=dict(family="Arial", size=13, color="black"),
        legend=dict(bgcolor="rgba(255,255,255,0.95)", bordercolor="black", borderwidth=1,
                    x=1.02, y=1.0, xanchor="left", yanchor="top"),
        scene=dict(
            bgcolor="white",
            xaxis=dict(title="X", showgrid=True, gridcolor="lightgray", zeroline=False,
                       showbackground=False, showline=True, linecolor="black", linewidth=1),
            yaxis=dict(title="Y", showgrid=True, gridcolor="lightgray", zeroline=False,
                       showbackground=False, showline=True, linecolor="black", linewidth=1),
            zaxis=dict(title="Z", showgrid=True, gridcolor="lightgray", zeroline=False,
                       showbackground=False, showline=True, linecolor="black", linewidth=1),
            aspectmode="cube", camera=dict(eye=dict(x=-1.6, y=-1.4, z=1.0)))
    )
    fig.show()

    # return results (same as before)
    if do_bifurcation:
        results_raw["hit_target"] = hit_target["name"] if hit_target else None
        results_raw["hit_target_idx"] = hit_target_idx
        results_raw["landing_point"] = landing_point
        results_raw["extrapolated_pts"] = np.array(extrapolated_pts) if extrapolated_pts else None
        results_raw["bifurcation_targets"] = bifurcation_targets
        results_raw["bifurcation_points"] = pos_raw[bif_idx_raw] if len(bif_idx_raw)>0 else np.empty((0,3))
        results_raw["bifurcations"] = bif_meta_raw
        return results_raw
    else:
        return {"positions": np.empty((0,3)), "distance": np.empty(0), "angles_deg": np.empty((0,0)),
                "bif_indices": [], "pair_corr": np.empty((0,0)), "min_corr": np.empty(0),
                "x_coords": np.empty(0), "hit_target": hit_target["name"] if hit_target else None,
                "hit_target_idx": hit_target_idx, "landing_point": landing_point,
                "extrapolated_pts": np.array(extrapolated_pts) if extrapolated_pts else None,
                "bifurcation_targets": target_centers, "bifurcation_points": np.empty((0,3)),
                "bifurcations": []}

Point at the smoothed master files, then run an example analysis:

In [34]:
MASTER_FILES_LOW = {
    "30.10.24": "../masters_csv/master_acoustic_trajectory_data_30Oct_low_smoothed.csv",
    "24.12.24": "../masters_csv/master_acoustic_trajectory_data_24Dec_low_smoothed.csv",
    "26.12.24": "../masters_csv/master_acoustic_trajectory_data_26Dec_low_smoothed.csv",
    "31.12.24": "../masters_csv/master_acoustic_trajectory_data_31Dec_low_smoothed.csv",
    "03.11.22": "../masters_csv/master_vision_trajectory_data_3Nov22_low_smoothed.csv",
    "01.10.22": "../masters_csv/master_vision_trajectory_data_1Oct22_low_smoothed.csv",
    "06.10.22": "../masters_csv/master_vision_trajectory_data_6Oct22_low_smoothed.csv",
    "27.10.22": "../masters_csv/master_vision_trajectory_data_27Oct22_low_smoothed.csv"
}

MASTER_FILES_HIGH = {
    "30.10.24": "../masters_csv/master_acoustic_trajectory_data_30Oct_high_smoothed.csv",
    "24.12.24": "../masters_csv/master_acoustic_trajectory_data_24Dec_high_smoothed.csv",
    "26.12.24": "../masters_csv/master_acoustic_trajectory_data_26Dec_high_smoothed.csv",
    "31.12.24": "../masters_csv/master_acoustic_trajectory_data_31Dec_high_smoothed.csv",
    "03.11.22": "../masters_csv/master_vision_trajectory_data_3Nov22_high_smoothed.csv",
    "01.10.22": "../masters_csv/master_vision_trajectory_data_1Oct22_high_smoothed.csv",
    "06.10.22": "../masters_csv/master_vision_trajectory_data_6Oct22_high_smoothed.csv",
    "27.10.22": "../masters_csv/master_vision_trajectory_data_27Oct22_high_smoothed.csv"
}


In [6]:
##interesting_trajectory:: pitt 33,38 , 26.12.24

In [42]:
results = analyze_trajectory_plotly(
    selected_date="01.10.22",
    selected_bat='ketem',
    selected_trajectory=68,
    required_hold_steps=30,
    window_size=30,
    min_corr_threshold=0.7,
    plateau_hold_steps=18,
    next_start=0,
    trajectory_fraction=0.7,
    angle_smoothing_window=61,
    angle_smoothing_polyorder=3,
    master_file_low=MASTER_FILES_LOW["26.12.24"],
    master_file_high=MASTER_FILES_HIGH["26.12.24"]
)

C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\1841020679.py:231: RuntimeWarning: All-NaN slice encountered
  min_corr_der  = np.nanmin(pair_corr_der, axis=1) if pair_corr_der.size>0 else np.empty(0)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\1841020679.py:263: RuntimeWarning: All-NaN slice encountered
  mcr = np.nanmin(pair_corr_raw, axis=1) if pair_corr_raw.size > 0 else np.empty(0)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\1841020679.py:459: RuntimeWarning: All-NaN slice encountered
  mcr = np.nanmin(pair_corr_raw, axis=1) if pair_corr_raw.size>0 else np.empty(0)


In [21]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import ast

# Define the vision dates based on your dataset
VISION_DATES = ["01.10.22", "06.10.22", "27.10.22", "03.11.22"]

# ============================================================
# Core per-trajectory computation (NO plotting)
# ============================================================

def _summarize_trajectory(
    date, bat, traj, df_low, df_high,
    target_centers, hitbox_targets,
    required_hold_steps, window_size, min_corr_threshold,
    plateau_hold_steps, next_start, trajectory_fraction,
    extrapolation_step, max_extrapolation,
    time_column, angle_smoothing_window, angle_smoothing_polyorder,
    start_skip
):
    mask_low = (df_low["bat_name"].astype(str) == bat) & (df_low["trajectory_no"] == traj)
    traj_low = df_low[mask_low].dropna(subset=["pos_x", "pos_y", "pos_z"])
    mask_high = (df_high["bat_name"].astype(str) == bat) & (df_high["trajectory_no"] == traj)
    traj_high = df_high[mask_high].dropna(subset=["pos_x", "pos_y", "pos_z"])

    if traj_low.empty or traj_high.empty:
        return None

    xs_low = traj_low[["pos_x", "pos_y", "pos_z"]].values[start_skip:]
    xs_high = traj_high[["pos_x", "pos_y", "pos_z"]].values[start_skip:]

    if len(xs_low) < 2 or len(xs_high) < 2:
        return None

    if time_column in traj_high.columns:
        time_full = traj_high[time_column].values
    else:
        time_full = np.arange(len(traj_high))

    # ---- landing classification (on xs_high) ----
    hit_target = None
    hit_target_idx = None
    hit_pt = None
    hit_idx = None
    for idx_pt, pt in enumerate(xs_high):
        for idx, ht in enumerate(hitbox_targets):
            if is_point_in_hitbox(pt, ht["box"]):
                hit_target = ht
                hit_target_idx = idx
                hit_pt = pt.copy()
                hit_idx = idx_pt
                break
        if hit_target:
            break

    if not hit_target:
        tail_len = max(2, int(len(xs_high) * 0.1))
        velocity = xs_high[-1] - xs_high[-tail_len]
        speed_val = np.linalg.norm(velocity)
        if speed_val > 0:
            direction = velocity / speed_val
            curr_pt = xs_high[-1].copy()
            for _ in range(max_extrapolation):
                curr_pt = curr_pt + direction * extrapolation_step
                for idx, ht in enumerate(hitbox_targets):
                    if is_point_in_hitbox(curr_pt, ht["box"]):
                        hit_target = ht
                        hit_target_idx = idx
                        hit_pt = curr_pt.copy()
                        break
                if hit_target:
                    break

    landing_point = None
    if hit_target and hit_pt is not None:
        box = hit_target["box"]
        c = box["center"]
        n = box["n"]
        v = hit_pt - c
        dist_to_plane = np.dot(v, n)
        landing_point = hit_pt - (dist_to_plane * n)

    # ---- truncation at landing ----
    if hit_idx is not None:
        xs_bif_low = xs_low[:hit_idx + 1]
        xs_bif_high = xs_high[:hit_idx + 1]
    else:
        xs_bif_low = xs_low
        xs_bif_high = xs_high
        
    # Modality categorization
    modality = "Vision" if date in VISION_DATES else "Acoustic"

    if len(xs_bif_high) < 2:
        return {
            "Date": date, "Modality": modality, "Bat Name": bat, "Trajectory No.": traj,
            "Target Hit": hit_target["name"] if hit_target else None,
            "No. of Bifurcations (Raw)": 0, "No. of Bifurcations (Deriv)": 0,
            "Location of 1st Bifurcation (Raw)": None, "Location of 1st Bifurcation (Deriv)": None,
            "Location of 2nd Bifurcation (Raw)": None, "Location of 2nd Bifurcation (Deriv)": None,
            "Angle to Targets at 1st Bifurcation (Raw)": None, "Angle to Targets at 1st Bifurcation (Deriv)": None,
            "Angle to Targets at 2nd Bifurcation (Raw)": None, "Angle to Targets at 2nd Bifurcation (Deriv)": None,
            "Typology of 1st Bifurcation (Raw)": None, "Typology of 1st Bifurcation (Deriv)": None,
            "Typology of 2nd Bifurcation (Raw)": None, "Typology of 2nd Bifurcation (Deriv)": None,
        }

    bifurcation_targets = target_centers.copy()
    if hit_target is not None and landing_point is not None:
        bifurcation_targets[hit_target_idx] = landing_point

    # ---- RAW detection (low-smoothed trajectory) ----
    results_raw = find_bifurcations(
        xs_bif_low, bifurcation_targets,
        required_hold_steps=required_hold_steps,
        window_size=window_size,
        min_corr_threshold=min_corr_threshold,
        plateau_hold_steps=plateau_hold_steps,
        next_start=next_start,
        trajectory_fraction=trajectory_fraction
    )
    pos_raw = results_raw["positions"]
    angles_low = results_raw["angles_deg"]
    bif_idx_raw = results_raw["bif_indices"]
    bif_meta_raw = results_raw.get("bifurcations", [])

    # ---- HIGH trajectory angles ----
    vel_high = np.diff(xs_bif_high, axis=0)
    vn_high = np.linalg.norm(vel_high, axis=1, keepdims=True)
    vu_high = vel_high / (vn_high + 1e-12)
    pos_high = xs_bif_high[:-1]

    n_targ = len(hitbox_targets)
    angles_high = np.zeros((len(pos_high), n_targ))
    for i, t in enumerate(target_centers):
        d = t - pos_high
        dn = np.linalg.norm(d, axis=1, keepdims=True)
        du = d / (dn + 1e-12)
        dots = np.sum(vu_high * du, axis=1)
        dots = np.clip(dots, -1, 1)
        angles_high[:, i] = np.degrees(np.arccos(dots))

    t_bif_high = time_full[:len(xs_bif_high)]
    t_axis_high = t_bif_high[:-1]

    data_len = len(angles_high)
    win = min(angle_smoothing_window, data_len - (1 if data_len % 2 == 0 else 0))
    if win <= angle_smoothing_polyorder:
        win = data_len - (1 if data_len % 2 == 0 else 0)
        poly = win - 1 if win > 1 else 0
    else:
        poly = angle_smoothing_polyorder

    if win > 2 and poly > 0:
        smooth_high = savgol_filter(angles_high, window_length=win, polyorder=poly, axis=0)
    else:
        smooth_high = angles_high

    d1_high = np.gradient(smooth_high, t_axis_high, axis=0)
    if win > 2 and poly > 0:
        d1_smooth = savgol_filter(d1_high, window_length=win, polyorder=poly, axis=0)
    else:
        d1_smooth = d1_high

    # ---- DERIV detection (on d1_smooth) ----
    results_der = find_bifurcations(
        xs_bif_high, bifurcation_targets,
        required_hold_steps=required_hold_steps,
        window_size=window_size,
        min_corr_threshold=min_corr_threshold,
        plateau_hold_steps=plateau_hold_steps,
        next_start=next_start,
        trajectory_fraction=trajectory_fraction,
        angles_deg=d1_smooth
    )
    bif_idx_der = results_der["bif_indices"]
    bif_meta_der = results_der.get("bifurcations", [])

    # ---- extract metadata per bifurcation ----
    def _pos(pos_arr, idx_list, k):
        if len(idx_list) > k:
            idx = idx_list[k]
            if idx < len(pos_arr):
                p = pos_arr[idx]
                return [round(float(p[0]), 3), round(float(p[1]), 3), round(float(p[2]), 3)]
        return None

    def _angles_at(idx_list, k):
        if len(idx_list) <= k:
            return None
        idx = idx_list[k]
        if idx >= len(angles_low):
            return None
        return [round(float(v), 3) for v in angles_low[idx]]
        
    def _typology_at(bif_meta_list, k):
        if len(bif_meta_list) > k:
            return bif_meta_list[k].get("typology", None)
        return None

    row = {
        "Date": date,
        "Modality": modality,
        "Bat Name": bat,
        "Trajectory No.": traj,
        "Target Hit": hit_target["name"] if hit_target else None,
        "No. of Bifurcations (Raw)": len(bif_idx_raw),
        "No. of Bifurcations (Deriv)": len(bif_idx_der),
        "Location of 1st Bifurcation (Raw)": _pos(pos_raw, bif_idx_raw, 0),
        "Location of 1st Bifurcation (Deriv)": _pos(pos_high, bif_idx_der, 0),
        "Location of 2nd Bifurcation (Raw)": _pos(pos_raw, bif_idx_raw, 1),
        "Location of 2nd Bifurcation (Deriv)": _pos(pos_high, bif_idx_der, 1),
        "Angle to Targets at 1st Bifurcation (Raw)": _angles_at(bif_idx_raw, 0),
        "Angle to Targets at 1st Bifurcation (Deriv)": _angles_at(bif_idx_der, 0),
        "Angle to Targets at 2nd Bifurcation (Raw)": _angles_at(bif_idx_raw, 1),
        "Angle to Targets at 2nd Bifurcation (Deriv)": _angles_at(bif_idx_der, 1),
        "Typology of 1st Bifurcation (Raw)": _typology_at(bif_meta_raw, 0),
        "Typology of 1st Bifurcation (Deriv)": _typology_at(bif_meta_der, 0),
        "Typology of 2nd Bifurcation (Raw)": _typology_at(bif_meta_raw, 1),
        "Typology of 2nd Bifurcation (Deriv)": _typology_at(bif_meta_der, 1),
    }
    return row

# ============================================================
# Batch driver
# ============================================================

def build_bifurcation_dataframe(
    dates=None,
    master_files_low=None,
    master_files_high=None,
    output_csv=None,
    required_hold_steps=30,
    window_size=25,
    min_corr_threshold=0.7,
    plateau_hold_steps=18,
    next_start=0,
    trajectory_fraction=0.7,
    hitbox_depth=200,
    hitbox_padding=50,
    extrapolation_step=10,
    max_extrapolation=200,
    time_column="Time",
    angle_smoothing_window=61,
    angle_smoothing_polyorder=3,
    start_skip=10
):
    if master_files_low is None:
        master_files_low = MASTER_FILES_LOW
    if master_files_high is None:
        master_files_high = MASTER_FILES_HIGH
    if dates is None:
        dates = sorted(set(master_files_low) & set(master_files_high))

    rows = []
    failures = []

    for date in dates:
        print(f"\nProcessing {date}...")

        df_low = pd.read_csv(master_files_low[date])
        df_low["trajectory_no"] = pd.to_numeric(df_low["trajectory_no"], errors="coerce")
        df_high = pd.read_csv(master_files_high[date])
        df_high["trajectory_no"] = pd.to_numeric(df_high["trajectory_no"], errors="coerce")

        pairs_low = set(map(tuple, df_low[["bat_name", "trajectory_no"]].dropna().values))
        pairs_high = set(map(tuple, df_high[["bat_name", "trajectory_no"]].dropna().values))
        pairs = sorted(pairs_low & pairs_high)

        rects = get_rectangles(date)
        target_centers = np.array([rectangle_center(rect) for rect in rects])
        hitbox_targets = [
            {"name": f"Rectangle {i+1}", "rect": rect,
             "color": None,
             "box": get_hitbox_params(rect, hitbox_depth, hitbox_padding)}
            for i, rect in enumerate(rects)
        ]

        for i, (bat, traj) in enumerate(pairs):
            bat = str(bat)
            try:
                row = _summarize_trajectory(
                    date, bat, traj, df_low, df_high,
                    target_centers, hitbox_targets,
                    required_hold_steps, window_size, min_corr_threshold,
                    plateau_hold_steps, next_start, trajectory_fraction,
                    extrapolation_step, max_extrapolation,
                    time_column, angle_smoothing_window, angle_smoothing_polyorder,
                    start_skip
                )
                if row is not None:
                    rows.append(row)
            except Exception as e:
                failures.append({"Date": date, "Bat Name": bat, "Trajectory No.": traj, "Error": str(e)})

            if (i + 1) % 25 == 0:
                print(f"  ...{i + 1}/{len(pairs)} trajectories done")

    df = pd.DataFrame(rows)

    if failures:
        print(f"\n{len(failures)} trajectories failed and were skipped:")
        for f in failures:
            print(f"  {f['Date']} | {f['Bat Name']} | traj {f['Trajectory No.']}: {f['Error']}")

    if output_csv:
        df.to_csv(output_csv, index=False)
        print(f"\nSaved: {output_csv}")

    return df, failures

In [22]:
# Run the batch processor across all dates and bats
df, failures = build_bifurcation_dataframe(
    output_csv="bifurcation_summary_all.csv",
    required_hold_steps=30,
    window_size=25,
    min_corr_threshold=0.7,
    plateau_hold_steps=18,
    next_start=0,
    trajectory_fraction=0.7,
    angle_smoothing_window=61,
    angle_smoothing_polyorder=3,
    start_skip=10
)

# Display the first few rows of the generated DataFrame right here in the notebook
display(df.head())


Processing 01.10.22...


C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN sli

  ...25/46 trajectories done


C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN sli


Processing 03.11.22...


...

  ...50/59 trajectories done


C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)



Saved: bifurcation_summary_all.csv


C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_3924\832228578.py:147: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)


,Date,Modality,Bat Name,Trajectory No.,Target Hit,No. of Bifurcations (Raw),No. of Bifurcations (Deriv),Location of 1st Bifurcation (Raw),Location of 1st Bifurcation (Deriv),Location of 2nd Bifurcation (Raw),Location of 2nd Bifurcation (Deriv),Angle to Targets at 1st Bifurcation (Raw),Angle to Targets at 1st Bifurcation (Deriv),Angle to Targets at 2nd Bifurcation (Raw),Angle to Targets at 2nd Bifurcation (Deriv),Typology of 1st Bifurcation (Raw),Typology of 1st Bifurcation (Deriv),Typology of 2nd Bifurcation (Raw),Typology of 2nd Bifurcation (Deriv)
0,01.10.22,Vision,ketem,23,Rectangle 2,0,0,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN
1,01.10.22,Vision,ketem,24,Rectangle 2,0,1,None,"[-205.521, -19.898, 688.512]",None,None,None,"[67.618, 10.944, 38.073]",None,None,NaN,edge-pair,NaN,NaN
2,01.10.22,Vision,ketem,31,Rectangle 2,1,0,"[110.543, 212.149, 817.726]",None,None,None,"[50.94, 23.819, 24.773]",None,None,None,edge-pair,NaN,NaN,NaN
3,01.10.22,Vision,ketem,56,Rectangle 2,1,1,"[64.862, 730.952, 675.704]","[-197.218, -257.686, 753.618]",None,None,"[35.449, 22.166, 15.22]","[77.669, 14.029, 44.239]",None,None,edge-pair,edge-pair,NaN,NaN
4,01.10.22,Vision,ketem,60,Rectangle 2,1,1,"[219.851, 800.88, 704.52]","[252.312, 1568.217, 808.073]",None,None,"[34.233, 22.029, 15.299]","[21.465, 33.182, 20.485]",None,None,edge-pair,edge-pair,NaN,NaN


In [23]:
df = pd.read_csv("bifurcation_summary_all.csv")

In [25]:
import pandas as pd

# Load the data if not already in memory
df = pd.read_csv("bifurcation_summary_all.csv")

# We only need a few columns to identify the trajectories
cols_to_show = ['Date', 'Bat Name', 'Trajectory No.', 'Target Hit']

print("🔍 HUNTING FOR TRAJECTORY EXAMPLES (Based on Raw Method)\n" + "="*60)

# 1) 2 bifurcations: 1st = edge-pair, 2nd = edge-center
cond1 = (df['No. of Bifurcations (Raw)'] == 2) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-pair') & \
        (df['Typology of 2nd Bifurcation (Raw)'] == 'edge-center')
res1 = df[cond1]
print(f"\n1) 2 Bifs [1st: edge-pair, 2nd: edge-center] --- Found: {len(res1)}")
if not res1.empty: display(res1[cols_to_show].head(3))

# 2) 2 bifurcations: 1st = edge-center, 2nd = edge-pair
cond2 = (df['No. of Bifurcations (Raw)'] == 2) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-center') & \
        (df['Typology of 2nd Bifurcation (Raw)'] == 'edge-pair')
res2 = df[cond2]
print(f"\n2) 2 Bifs [1st: edge-center, 2nd: edge-pair] --- Found: {len(res2)}")
if not res2.empty: display(res2[cols_to_show].head(3))

# 3) 2 bifurcations: both edge-center
cond3 = (df['No. of Bifurcations (Raw)'] == 2) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-center') & \
        (df['Typology of 2nd Bifurcation (Raw)'] == 'edge-center')
res3 = df[cond3]
print(f"\n3) 2 Bifs [Both edge-center] --- Found: {len(res3)}")
if not res3.empty: display(res3[cols_to_show].head(3))

# 4) 2 bifurcations: both edge-pair
cond4 = (df['No. of Bifurcations (Raw)'] == 2) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-pair') & \
        (df['Typology of 2nd Bifurcation (Raw)'] == 'edge-pair')
res4 = df[cond4]
print(f"\n4) 2 Bifs [Both edge-pair] --- Found: {len(res4)}")
if not res4.empty: display(res4[cols_to_show].head(3))

# 5) 1 bifurcation: edge-pair
cond5 = (df['No. of Bifurcations (Raw)'] == 1) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-pair')
res5 = df[cond5]
print(f"\n5) 1 Bif [edge-pair] --- Found: {len(res5)}")
if not res5.empty: display(res5[cols_to_show].head(3))

# 6) 1 bifurcation: edge-center
cond6 = (df['No. of Bifurcations (Raw)'] == 1) & \
        (df['Typology of 1st Bifurcation (Raw)'] == 'edge-center')
res6 = df[cond6]
print(f"\n6) 1 Bif [edge-center] --- Found: {len(res6)}")
if not res6.empty: display(res6[cols_to_show].head(3))

🔍 HUNTING FOR TRAJECTORY EXAMPLES (Based on Raw Method)

1) 2 Bifs [1st: edge-pair, 2nd: edge-center] --- Found: 7


,Date,Bat Name,Trajectory No.,Target Hit
94,03.11.22,pitt,41,Rectangle 3
163,24.12.24,pitt,22,Rectangle 3
182,24.12.24,pitt,41,NaN



2) 2 Bifs [1st: edge-center, 2nd: edge-pair] --- Found: 3


,Date,Bat Name,Trajectory No.,Target Hit
423,31.12.24,ketem,13,Rectangle 2
438,31.12.24,ketem,28,Rectangle 2
460,31.12.24,ketem,51,Rectangle 1



3) 2 Bifs [Both edge-center] --- Found: 5


...

,Date,Bat Name,Trajectory No.,Target Hit
32,01.10.22,ketem,102,Rectangle 1
48,03.11.22,ketem,3,Rectangle 2
52,03.11.22,ketem,8,Rectangle 1



5) 1 Bif [edge-pair] --- Found: 263


,Date,Bat Name,Trajectory No.,Target Hit
2,01.10.22,ketem,31,Rectangle 2
3,01.10.22,ketem,56,Rectangle 2
4,01.10.22,ketem,60,Rectangle 2



6) 1 Bif [edge-center] --- Found: 20


,Date,Bat Name,Trajectory No.,Target Hit
15,01.10.22,ketem,79,Rectangle 3
36,01.10.22,ketem,107,Rectangle 3
44,01.10.22,ketem,116,Rectangle 3


In [26]:
import pandas as pd

# Load the data if not already in memory
# df = pd.read_csv("bifurcation_summary_all.csv")

# We only need a few columns to identify the trajectories
cols_to_show = ['Date', 'Bat Name', 'Trajectory No.', 'Target Hit']

print("🔍 HUNTING FOR TRAJECTORY EXAMPLES (Derivative Method)\n" + "="*60)

# 1) 2 bifurcations: 1st = edge-pair, 2nd = edge-center
cond1 = (df['No. of Bifurcations (Deriv)'] == 2) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-pair') & \
        (df['Typology of 2nd Bifurcation (Deriv)'] == 'edge-center')
res1 = df[cond1]
print(f"\n1) 2 Bifs [1st: edge-pair, 2nd: edge-center] --- Found: {len(res1)}")
if not res1.empty: display(res1[cols_to_show].head(3))

# 2) 2 bifurcations: 1st = edge-center, 2nd = edge-pair
cond2 = (df['No. of Bifurcations (Deriv)'] == 2) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-center') & \
        (df['Typology of 2nd Bifurcation (Deriv)'] == 'edge-pair')
res2 = df[cond2]
print(f"\n2) 2 Bifs [1st: edge-center, 2nd: edge-pair] --- Found: {len(res2)}")
if not res2.empty: display(res2[cols_to_show].head(3))

# 3) 2 bifurcations: both edge-center
cond3 = (df['No. of Bifurcations (Deriv)'] == 2) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-center') & \
        (df['Typology of 2nd Bifurcation (Deriv)'] == 'edge-center')
res3 = df[cond3]
print(f"\n3) 2 Bifs [Both edge-center] --- Found: {len(res3)}")
if not res3.empty: display(res3[cols_to_show].head(3))

# 4) 2 bifurcations: both edge-pair
cond4 = (df['No. of Bifurcations (Deriv)'] == 2) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-pair') & \
        (df['Typology of 2nd Bifurcation (Deriv)'] == 'edge-pair')
res4 = df[cond4]
print(f"\n4) 2 Bifs [Both edge-pair] --- Found: {len(res4)}")
if not res4.empty: display(res4[cols_to_show].head(3))

# 5) 1 bifurcation: edge-pair
cond5 = (df['No. of Bifurcations (Deriv)'] == 1) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-pair')
res5 = df[cond5]
print(f"\n5) 1 Bif [edge-pair] --- Found: {len(res5)}")
if not res5.empty: display(res5[cols_to_show].head(3))

# 6) 1 bifurcation: edge-center
cond6 = (df['No. of Bifurcations (Deriv)'] == 1) & \
        (df['Typology of 1st Bifurcation (Deriv)'] == 'edge-center')
res6 = df[cond6]
print(f"\n6) 1 Bif [edge-center] --- Found: {len(res6)}")
if not res6.empty: display(res6[cols_to_show].head(3))

🔍 HUNTING FOR TRAJECTORY EXAMPLES (Derivative Method)

1) 2 Bifs [1st: edge-pair, 2nd: edge-center] --- Found: 30


,Date,Bat Name,Trajectory No.,Target Hit
7,01.10.22,ketem,63,Rectangle 1
9,01.10.22,ketem,65,Rectangle 1
50,03.11.22,ketem,6,Rectangle 1



2) 2 Bifs [1st: edge-center, 2nd: edge-pair] --- Found: 33


,Date,Bat Name,Trajectory No.,Target Hit
5,01.10.22,ketem,61,Rectangle 1
14,01.10.22,ketem,78,Rectangle 2
46,03.11.22,ketem,1,Rectangle 3



3) 2 Bifs [Both edge-center] --- Found: 11


...

,Date,Bat Name,Trajectory No.,Target Hit
10,01.10.22,ketem,68,Rectangle 2
11,01.10.22,ketem,70,Rectangle 2
16,01.10.22,ketem,80,Rectangle 2



5) 1 Bif [edge-pair] --- Found: 177


,Date,Bat Name,Trajectory No.,Target Hit
1,01.10.22,ketem,24,Rectangle 2
3,01.10.22,ketem,56,Rectangle 2
4,01.10.22,ketem,60,Rectangle 2



6) 1 Bif [edge-center] --- Found: 60


,Date,Bat Name,Trajectory No.,Target Hit
15,01.10.22,ketem,79,Rectangle 3
19,01.10.22,ketem,84,Rectangle 1
32,01.10.22,ketem,102,Rectangle 1


In [27]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Load the dataset
df = pd.read_csv("bifurcation_summary_all.csv")

# Filter out the 2-target day
df_filtered = df[df['Date'] != '06.10.22'].copy()

def classify_trajectory(row, method):
    n_bif = row[f'No. of Bifurcations ({method})']
    if pd.isna(n_bif) or n_bif == 0:
        return '0 Bifurcations'
    
    t1 = row.get(f'Typology of 1st Bifurcation ({method})', None)
    t2 = row.get(f'Typology of 2nd Bifurcation ({method})', None)
    
    if n_bif == 1:
        if t1 == 'edge-pair': return '1 Bif: Edge-Pair'
        if t1 == 'edge-center': return '1 Bif: Edge-Center'
        return '1 Bif: Other/Unknown'
    
    if n_bif == 2:
        if t1 == 'edge-pair' and t2 == 'edge-center': return '2 Bifs: Edge-Pair ➔ Edge-Center'
        if t1 == 'edge-center' and t2 == 'edge-pair': return '2 Bifs: Edge-Center ➔ Edge-Pair'
        if t1 == 'edge-center' and t2 == 'edge-center': return '2 Bifs: Edge-Center ➔ Edge-Center'
        if t1 == 'edge-pair' and t2 == 'edge-pair': return '2 Bifs: Edge-Pair ➔ Edge-Pair'
        return '2 Bifs: Other/Unknown'
        
    return '3+ Bifurcations'

# Apply classifications
df_filtered['Class_Raw'] = df_filtered.apply(lambda r: classify_trajectory(r, 'Raw'), axis=1)
df_filtered['Class_Deriv'] = df_filtered.apply(lambda r: classify_trajectory(r, 'Deriv'), axis=1)

# Count the occurrences
counts_raw = df_filtered['Class_Raw'].value_counts()
counts_deriv = df_filtered['Class_Deriv'].value_counts()

# Create side-by-side Pie Charts
fig = make_subplots(
    rows=1, cols=2, 
    specs=[[{'type':'domain'}, {'type':'domain'}]],
    subplot_titles=["Raw Method Typology", "Derivative Method Typology"]
)

# Custom template to show Name, Value, and Percentage
text_temp = "%{label}<br><b>%{value}</b> traj (%{percent})"

fig.add_trace(
    go.Pie(
        labels=counts_raw.index, 
        values=counts_raw.values,
        texttemplate=text_temp,
        name="Raw",
        hoverinfo="label+value+percent"
    ), 
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=counts_deriv.index, 
        values=counts_deriv.values,
        texttemplate=text_temp,
        name="Deriv",
        hoverinfo="label+value+percent"
    ), 
    row=1, col=2
)

fig.update_layout(
    title=dict(text="Trajectory Classification by Typology (Excl. 06.10.22)", x=0.5, font=dict(size=22)),
    height=750,
    width=1400,
    showlegend=True,
    legend=dict(yanchor="top", y=1, xanchor="left", x=1.05)
)

fig.show()

In [28]:
import pandas as pd
import numpy as np
import ast
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Load the dataset and filter
df = pd.read_csv("bifurcation_summary_all.csv")
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Safe parser for the angle lists stored as strings in the CSV
def parse_angles(val):
    if pd.isna(val): return None
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except (ValueError, SyntaxError): return None
    if isinstance(val, (list, np.ndarray)): return val
    return None

# Parse all relevant angle columns back into Python lists
for m in ['Raw', 'Deriv']:
    df_filtered[f'Angle to Targets at 1st Bifurcation ({m})'] = df_filtered[f'Angle to Targets at 1st Bifurcation ({m})'].apply(parse_angles)
    df_filtered[f'Angle to Targets at 2nd Bifurcation ({m})'] = df_filtered[f'Angle to Targets at 2nd Bifurcation ({m})'].apply(parse_angles)

target_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

# Generator function for the distribution pages
def plot_typology_angles(method, subset_name, subset_df):
    if subset_df.empty: return
    
    # Aggregation buckets for the angles
    angles = {
        'edge-pair': [[], [], []],   # [Target 1 angles, Target 2 angles, Target 3 angles]
        'edge-center': [[], [], []]
    }
    
    # Aggregate angles based on the typology tag (combining both 1st and 2nd bifurcations)
    for _, row in subset_df.iterrows():
        for prefix in ['1st', '2nd']:
            typ = row.get(f'Typology of {prefix} Bifurcation ({method})')
            angs = row.get(f'Angle to Targets at {prefix} Bifurcation ({method})')
            
            if typ in angles and angs is not None and len(angs) >= 3:
                angles[typ][0].append(angs[0])
                angles[typ][1].append(angs[1])
                angles[typ][2].append(angs[2])
                
    titles = [
        "Edge-Pair | Target 1", "Edge-Pair | Target 2", "Edge-Pair | Target 3",
        "Edge-Center | Target 1", "Edge-Center | Target 2", "Edge-Center | Target 3"
    ]
    
    fig = make_subplots(
        rows=2, cols=3, 
        subplot_titles=titles, 
        vertical_spacing=0.15,
        horizontal_spacing=0.05
    )
    
    row_map = {'edge-pair': 1, 'edge-center': 2}
    
    for typ in ['edge-pair', 'edge-center']:
        r = row_map[typ]
        for t_idx in range(3):
            c = t_idx + 1
            data = angles[typ][t_idx]
            
            if data:
                fig.add_trace(
                    go.Histogram(
                        x=data, 
                        marker_color=target_colors[t_idx],
                        xbins=dict(start=0, end=180, size=5), 
                        showlegend=False
                    ), 
                    row=r, col=c
                )
                
                # Draw the mean line
                mean_val = np.mean(data)
                fig.add_vline(
                    x=mean_val, line_dash="dash", line_color="black", line_width=2,
                    annotation_text=f"Mean: {mean_val:.1f}°", annotation_position="top right",
                    row=r, col=c
                )
                
            # Clean up axes
            fig.update_xaxes(title_text="Angle (°)", range=[0, 180], row=r, col=c)
            if c == 1: 
                fig.update_yaxes(title_text="Count", row=r, col=c)
            
    fig.update_layout(
        title=dict(text=f"Bifurcation Angles by Typology: {method} Method | {subset_name} (Excl. 06.10.22)", x=0.5),
        height=800, width=1400, 
        template="simple_white"
    )
    fig.show()

# ----------------------------------------------------
# Generate all 6 plot pages
# ----------------------------------------------------
for method in ['Raw', 'Deriv']:
    # 1. All Days
    plot_typology_angles(method, "All Days", df_filtered)
    
    # 2. Vision Days
    plot_typology_angles(method, "Vision Days", df_filtered[df_filtered['Modality'] == 'Vision'])
    
    # 3. Acoustic Days
    plot_typology_angles(method, "Acoustic Days", df_filtered[df_filtered['Modality'] == 'Acoustic'])

In [10]:
import pandas as pd
import numpy as np
import ast
import plotly.express as px

# Helper to safely parse coordinate lists (handles both in-memory lists and CSV-loaded strings)
def parse_pt(pt):
    if pt is None:
        return None
    # If it's already a list or numpy array, return it directly
    if isinstance(pt, (list, np.ndarray)):
        return np.array(pt)
    # If it's a string, evaluate it back into a list
    if isinstance(pt, str):
        try:
            parsed = ast.literal_eval(pt)
            return np.array(parsed)
        except (ValueError, SyntaxError):
            return None
    # Catch stray NaNs (which are floats)
    if isinstance(pt, float) and np.isnan(pt):
        return None
    
    return None

# Helper to calculate 3D Euclidean distance
def calc_dist(pt1, pt2):
    p1 = parse_pt(pt1)
    p2 = parse_pt(pt2)
    if p1 is not None and p2 is not None:
        return np.linalg.norm(p1 - p2)
    return np.nan

# ==========================================
# 1. Percentage of Trajectories with Mismatch
# ==========================================
total_trajs = len(df)
df['Agree'] = df['No. of Bifurcations (Raw)'] == df['No. of Bifurcations (Deriv)']
disagree_pct = (~df['Agree']).mean() * 100

print("==================================================")
print(f"1) Mismatch Percentage: {disagree_pct:.2f}% of trajectories ({(~df['Agree']).sum()}/{total_trajs}) do not agree.")
print("==================================================\n")

# ==========================================
# 2. Pie Chart of Mismatch Types
# ==========================================
df_mismatch = df[~df['Agree']].copy()

# Create a clean label for the pie chart slices
df_mismatch['Mismatch Type'] = (
    "Raw: " + df_mismatch['No. of Bifurcations (Raw)'].astype(int).astype(str) + 
    " | Deriv: " + df_mismatch['No. of Bifurcations (Deriv)'].astype(int).astype(str)
)

fig = px.pie(
    df_mismatch, 
    names='Mismatch Type', 
    title='Distribution of Mismatch Types (Where Methods Disagree)',
    hole=0.3,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

# ==========================================
# 3. Average Distance for Agreeing Trajectories
# ==========================================
# Calculate distances for all rows first
df['Dist_1st'] = df.apply(lambda row: calc_dist(row['Location of 1st Bifurcation (Raw)'], row['Location of 1st Bifurcation (Deriv)']), axis=1)
df['Dist_2nd'] = df.apply(lambda row: calc_dist(row['Location of 2nd Bifurcation (Raw)'], row['Location of 2nd Bifurcation (Deriv)']), axis=1)

# Group A: Agree on exactly 1 bifurcation
df_agree_1 = df[(df['Agree']) & (df['No. of Bifurcations (Raw)'] == 1)]
avg_dist_1_only = df_agree_1['Dist_1st'].mean()

# Group B: Agree on exactly 2 bifurcations
df_agree_2 = df[(df['Agree']) & (df['No. of Bifurcations (Raw)'] == 2)]
avg_dist_1_for_2 = df_agree_2['Dist_1st'].mean()
avg_dist_2_for_2 = df_agree_2['Dist_2nd'].mean()

print("==================================================")
print("3) Average 3D Distances (For Agreeing Trajectories)")
print("==================================================")
print(f"For trajectories with exactly 1 bifurcation (Count: {len(df_agree_1)}):")
print(f"  -> Avg distance between 1st points: {avg_dist_1_only:.2f} mm\n")

print(f"For trajectories with exactly 2 bifurcations (Count: {len(df_agree_2)}):")
print(f"  -> Avg distance between 1st points: {avg_dist_1_for_2:.2f} mm")
print(f"  -> Avg distance between 2nd points: {avg_dist_2_for_2:.2f} mm")
print("==================================================")

1) Mismatch Percentage: 57.32% of trajectories (270/471) do not agree.

3) Average 3D Distances (For Agreeing Trajectories)
For trajectories with exactly 1 bifurcation (Count: 143):
  -> Avg distance between 1st points: 574.31 mm

For trajectories with exactly 2 bifurcations (Count: 38):
  -> Avg distance between 1st points: 324.61 mm
  -> Avg distance between 2nd points: 655.81 mm


In [15]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Standardize bat names to lowercase to avoid splitting 'Pitt' and 'pitt' into separate groups
df['Bat Name'] = df['Bat Name'].astype(str).str.lower()

# Group by Date and Bat Name
grouped = df.groupby(['Date', 'Bat Name'])
combos = list(grouped.groups.keys())

# Sort true chronologically (parsing DD.MM.YY) and then alphabetically by bat
combos.sort(key=lambda x: (pd.to_datetime(x[0], format='%d.%m.%y'), x[1]))

num_rows = len(combos)

# Prepare dynamic subplot titles for 3 columns
titles = []
for date, bat in combos:
    titles.append(f"{date} | {bat.capitalize()} - Target Hit")
    titles.append(f"{date} | {bat.capitalize()} - Bif (Raw)")
    titles.append(f"{date} | {bat.capitalize()} - Bif (Deriv)")

# Create a figure with dynamic rows and 3 columns
fig = make_subplots(
    rows=num_rows, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.03 # Tight spacing for easier scrolling
)

for i, (date, bat) in enumerate(combos, start=1):
    subset = grouped.get_group((date, bat))
    
    # ----------------------------------------------------
    # 1. Target Hit Distribution (Percentage)
    # ----------------------------------------------------
    # normalize=True gives proportions (0.0 to 1.0), multiply by 100 for percentages
    target_pct = subset['Target Hit'].fillna('No Hit').value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=target_pct.index.astype(str),
            y=target_pct.values,
            marker_color="#1f77b4",  # Blue
            text=[f"{val:.1f}%" for val in target_pct.values],
            textposition='auto'
        ),
        row=i, col=1
    )
    
    # ----------------------------------------------------
    # 2. Bifurcation Distribution (Raw - Percentage)
    # ----------------------------------------------------
    bif_pct_raw = subset['No. of Bifurcations (Raw)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_raw.index.astype(str),
            y=bif_pct_raw.values,
            marker_color="#ff7f0e",  # Orange
            text=[f"{val:.1f}%" for val in bif_pct_raw.values],
            textposition='auto'
        ),
        row=i, col=2
    )

    # ----------------------------------------------------
    # 3. Bifurcation Distribution (Deriv - Percentage)
    # ----------------------------------------------------
    bif_pct_deriv = subset['No. of Bifurcations (Deriv)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_deriv.index.astype(str),
            y=bif_pct_deriv.values,
            marker_color="#2ca02c",  # Green
            text=[f"{val:.1f}%" for val in bif_pct_deriv.values],
            textposition='auto'
        ),
        row=i, col=3
    )
    
    # Clean up axes for all three columns
    fig.update_xaxes(title_text="Target", type='category', row=i, col=1)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=1)
    
    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=2)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=2)

    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=3)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=3)

# ----------------------------------------------------
# Draw thick horizontal lines between different days
# ----------------------------------------------------
prev_date = None
date_change_rows = []

# Find the rows where the date transitions
for i, (date, bat) in enumerate(combos, start=1):
    if prev_date is not None and date != prev_date:
        date_change_rows.append(i - 1)  # Record the row right before the change
    prev_date = date

# Draw the lines exactly between the subplots
for r in date_change_rows:
    # Get Plotly's internal y-axis names for the row above and the row below
    y_axis_above = 'yaxis' if r == 1 else f'yaxis{(r-1)*3 + 1}'
    y_axis_below = f'yaxis{r*3 + 1}'
    
    # Extract the domain borders to find the perfect visual midpoint
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",  # Span the entire width of the page
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=4)
    )

# Scale height dynamically: ~350 pixels per row ensures the subplots don't squish
total_height = max(800, 350 * num_rows)

fig.update_layout(
    title=dict(
        text="Daily Trajectory Analysis per Bat (%)",
        x=0.5, font=dict(size=22)
    ),
    height=total_height,
    width=1600,
    template="simple_white",
    showlegend=False
)

fig.show()

In [17]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Standardize bat names to lowercase
df['Bat Name'] = df['Bat Name'].astype(str).str.lower()

# Filter out the 2-target day
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Group by Bat Name ONLY
grouped = df_filtered.groupby('Bat Name')
bats = list(grouped.groups.keys())
bats.sort()  # Sort alphabetically by bat

num_rows = len(bats)

# Prepare dynamic subplot titles for 3 columns
titles = []
for bat in bats:
    titles.append(f"{bat.capitalize()} - Target Hit")
    titles.append(f"{bat.capitalize()} - Bif (Raw)")
    titles.append(f"{bat.capitalize()} - Bif (Deriv)")

# Create a figure with dynamic rows and 3 columns
fig = make_subplots(
    rows=num_rows, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.12 # Increased spacing to prevent titles from overlapping
)

for i, bat in enumerate(bats, start=1):
    subset = grouped.get_group(bat)
    
    # ----------------------------------------------------
    # 1. Target Hit Distribution (Percentage)
    # ----------------------------------------------------
    target_pct = subset['Target Hit'].fillna('No Hit').value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=target_pct.index.astype(str),
            y=target_pct.values,
            marker_color="#1f77b4",  # Blue
            text=[f"{val:.1f}%" for val in target_pct.values],
            textposition='auto'
        ),
        row=i, col=1
    )
    
    # ----------------------------------------------------
    # 2. Bifurcation Distribution (Raw - Percentage)
    # ----------------------------------------------------
    bif_pct_raw = subset['No. of Bifurcations (Raw)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_raw.index.astype(str),
            y=bif_pct_raw.values,
            marker_color="#ff7f0e",  # Orange
            text=[f"{val:.1f}%" for val in bif_pct_raw.values],
            textposition='auto'
        ),
        row=i, col=2
    )

    # ----------------------------------------------------
    # 3. Bifurcation Distribution (Deriv - Percentage)
    # ----------------------------------------------------
    bif_pct_deriv = subset['No. of Bifurcations (Deriv)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_deriv.index.astype(str),
            y=bif_pct_deriv.values,
            marker_color="#2ca02c",  # Green
            text=[f"{val:.1f}%" for val in bif_pct_deriv.values],
            textposition='auto'
        ),
        row=i, col=3
    )
    
    # Clean up axes for all three columns
    fig.update_xaxes(title_text="Target", type='category', row=i, col=1)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=1)
    
    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=2)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=2)

    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=3)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=3)

# ----------------------------------------------------
# Draw horizontal lines between different bats
# ----------------------------------------------------
for r in range(1, num_rows):
    # Get Plotly's internal y-axis names for the row above and the row below
    y_axis_above = 'yaxis' if r == 1 else f'yaxis{(r-1)*3 + 1}'
    y_axis_below = f'yaxis{r*3 + 1}'
    
    # Extract the domain borders to find the perfect visual midpoint
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",  # Span the entire width of the page
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=2)
    )

# Scale height dynamically: Increased to 450 pixels per row to maintain plot sizes
total_height = max(800, 450 * num_rows)

fig.update_layout(
    title=dict(
        text="Overall Trajectory Analysis per Bat (Excl. 06.10.22) (%)",
        x=0.5, font=dict(size=22)
    ),
    height=total_height,
    width=1600,
    template="simple_white",
    showlegend=False
)

fig.show()

In [21]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Filter out the 2-target day
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Define the 3 subsets for the rows
datasets = [
    ("All Days", df_filtered),
    ("Vision Days", df_filtered[df_filtered['Modality'] == 'Vision']),
    ("Acoustic Days", df_filtered[df_filtered['Modality'] == 'Acoustic'])
]

num_rows = len(datasets)

# Prepare dynamic subplot titles for 3 columns
titles = []
for name, _ in datasets:
    titles.append(f"{name} - Target Hit")
    titles.append(f"{name} - Bif (Raw)")
    titles.append(f"{name} - Bif (Deriv)")

# Create a figure with dynamic rows and 3 columns
fig = make_subplots(
    rows=num_rows, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.15 # Increased spacing to prevent titles from overlapping
)

for i, (name, subset) in enumerate(datasets, start=1):
    # Safety check in case a modality is missing
    if subset.empty:
        continue
        
    # ----------------------------------------------------
    # 1. Target Hit Distribution (Percentage)
    # ----------------------------------------------------
    target_pct = subset['Target Hit'].fillna('No Hit').value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=target_pct.index.astype(str),
            y=target_pct.values,
            marker_color="#1f77b4",  # Blue
            text=[f"{val:.1f}%" for val in target_pct.values],
            textposition='auto'
        ),
        row=i, col=1
    )
    
    # ----------------------------------------------------
    # 2. Bifurcation Distribution (Raw - Percentage)
    # ----------------------------------------------------
    bif_pct_raw = subset['No. of Bifurcations (Raw)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_raw.index.astype(str),
            y=bif_pct_raw.values,
            marker_color="#ff7f0e",  # Orange
            text=[f"{val:.1f}%" for val in bif_pct_raw.values],
            textposition='auto'
        ),
        row=i, col=2
    )

    # ----------------------------------------------------
    # 3. Bifurcation Distribution (Deriv - Percentage)
    # ----------------------------------------------------
    bif_pct_deriv = subset['No. of Bifurcations (Deriv)'].value_counts(normalize=True).sort_index() * 100
    
    fig.add_trace(
        go.Bar(
            x=bif_pct_deriv.index.astype(str),
            y=bif_pct_deriv.values,
            marker_color="#2ca02c",  # Green
            text=[f"{val:.1f}%" for val in bif_pct_deriv.values],
            textposition='auto'
        ),
        row=i, col=3
    )
    
    # Clean up axes for all three columns
    fig.update_xaxes(title_text="Target", type='category', row=i, col=1)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=1)
    
    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=2)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=2)

    fig.update_xaxes(title_text="No. of Bifurcations", type='category', row=i, col=3)
    fig.update_yaxes(title_text="Percentage (%)", range=[0, 105], row=i, col=3)

# ----------------------------------------------------
# Draw horizontal lines between different rows
# ----------------------------------------------------
for r in range(1, num_rows):
    # Get Plotly's internal y-axis names for the row above and the row below
    y_axis_above = 'yaxis' if r == 1 else f'yaxis{(r-1)*3 + 1}'
    y_axis_below = f'yaxis{r*3 + 1}'
    
    # Extract the domain borders to find the perfect visual midpoint
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",  # Span the entire width of the page
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=2)
    )

# Scale height dynamically
total_height = max(800, 450 * num_rows)

fig.update_layout(
    title=dict(
        text="Overall Trajectory Analysis by Modality (Excl. 06.10.22) (%)",
        x=0.5, font=dict(size=22)
    ),
    height=total_height,
    width=1600,
    template="simple_white",
    showlegend=False
)

fig.show()

In [22]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px

pio.renderers.default = "browser"

# Standardize bat names to lowercase
df['Bat Name'] = df['Bat Name'].astype(str).str.lower()
df['Target Hit'] = df['Target Hit'].fillna('No Hit')

# Create the 10-trajectory chunks
# Chunk 0 = trajs 1-10, Chunk 1 = trajs 11-20, etc.
df['Chunk_Int'] = (df['Trajectory No.'] - 1) // 10
df['Chunk_Label'] = (df['Chunk_Int'] * 10 + 1).astype(str) + '-' + (df['Chunk_Int'] * 10 + 10).astype(str)

# Group by Date and Bat Name
grouped = df.groupby(['Date', 'Bat Name'])
combos = list(grouped.groups.keys())
combos.sort(key=lambda x: (pd.to_datetime(x[0], format='%d.%m.%y'), x[1]))

num_rows = len(combos)

# Prepare dynamic subplot titles
titles = []
for date, bat in combos:
    titles.append(f"{date} | {bat.capitalize()} - Target Hit %")
    titles.append(f"{date} | {bat.capitalize()} - Bif (Raw) %")
    titles.append(f"{date} | {bat.capitalize()} - Bif (Deriv) %")

fig = make_subplots(
    rows=num_rows, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.05,
    vertical_spacing=0.04
)

# Define consistent colors across all plots
all_targets = sorted(df['Target Hit'].unique().tolist())
target_colors = {t: px.colors.qualitative.Safe[i % len(px.colors.qualitative.Safe)] for i, t in enumerate(all_targets)}
bif_colors = {0: "#d62728", 1: "#1f77b4", 2: "#2ca02c"}  # Red for 0, Blue for 1, Green for 2

# Keep track of which legend items have been added so we don't duplicate them
added_to_legend = set()

for i, (date, bat) in enumerate(combos, start=1):
    subset = grouped.get_group((date, bat))
    
    # ----------------------------------------------------
    # Calculate Percentages per Chunk
    # ----------------------------------------------------
    # normalize='index' calculates the % across each row (chunk)
    target_xtab = pd.crosstab(subset['Chunk_Int'], subset['Target Hit'], normalize='index') * 100
    bif_raw_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Raw)'], normalize='index') * 100
    bif_deriv_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Deriv)'], normalize='index') * 100
    
    # Ensure all possible columns exist (so lines don't break if 0% happen in a whole day)
    target_xtab = target_xtab.reindex(columns=all_targets, fill_value=0.0)
    bif_raw_xtab = bif_raw_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    bif_deriv_xtab = bif_deriv_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    
    # Generate ordered x-axis labels based on the integer chunks present for this bat
    x_vals = target_xtab.index
    x_labels = [f"{c*10+1}-{c*10+10}" for c in x_vals]
    
    # ----------------------------------------------------
    # 1. Target Hit Line Plot
    # ----------------------------------------------------
    for target in all_targets:
        show_leg = (target not in added_to_legend)
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=target_xtab[target].values,
                mode='lines+markers',
                name=target,
                legendgroup='targets',
                legendgrouptitle_text="Targets",
                line=dict(color=target_colors[target], width=2),
                marker=dict(size=8),
                showlegend=show_leg
            ),
            row=i, col=1
        )
        added_to_legend.add(target)
        
    # ----------------------------------------------------
    # 2. Bifurcation Line Plot (Raw)
    # ----------------------------------------------------
    for bif_count in [0, 1, 2]:
        name_str = f"{bif_count} Bifurcation{'s' if bif_count != 1 else ''}"
        show_leg = (name_str not in added_to_legend)
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_raw_xtab[bif_count].values,
                mode='lines+markers',
                name=name_str,
                legendgroup='bifurcations',
                legendgrouptitle_text="Bifurcations",
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=show_leg
            ),
            row=i, col=2
        )
        added_to_legend.add(name_str)

    # ----------------------------------------------------
    # 3. Bifurcation Line Plot (Deriv)
    # ----------------------------------------------------
    for bif_count in [0, 1, 2]:
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_deriv_xtab[bif_count].values,
                mode='lines+markers',
                name=f"{bif_count} Bifurcation{'s' if bif_count != 1 else ''}",
                legendgroup='bifurcations',
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=False  # Already handled in col 2
            ),
            row=i, col=3
        )
        
    # Clean up axes for all three columns
    for col in [1, 2, 3]:
        fig.update_xaxes(title_text="Trajectory Chunk", type='category', row=i, col=col)
        fig.update_yaxes(title_text="% of Trajectories", range=[-5, 105], row=i, col=col)

# ----------------------------------------------------
# Draw thick horizontal lines between different days
# ----------------------------------------------------
prev_date = None
date_change_rows = []

for i, (date, bat) in enumerate(combos, start=1):
    if prev_date is not None and date != prev_date:
        date_change_rows.append(i - 1)
    prev_date = date

for r in date_change_rows:
    y_axis_above = 'yaxis' if r == 1 else f'yaxis{(r-1)*3 + 1}'
    y_axis_below = f'yaxis{r*3 + 1}'
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=4)
    )

total_height = max(800, 400 * num_rows)

fig.update_layout(
    title=dict(
        text="Temporal Bias: Daily Trajectory Analysis per Bat",
        x=0.5, font=dict(size=22)
    ),
    height=total_height,
    width=1700,
    template="simple_white",
    hovermode="x unified",  # Shows all values for a chunk when you hover over it
    legend=dict(
        yanchor="top", y=1, xanchor="left", x=1.02,
        bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
    )
)

fig.show()

In [20]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px

pio.renderers.default = "browser"

# Standardize bat names to lowercase
df['Bat Name'] = df['Bat Name'].astype(str).str.lower()
df['Target Hit'] = df['Target Hit'].fillna('No Hit')

# Filter out the 2-target day to keep the target list consistent
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Create the 10-trajectory chunks
# Chunk 0 = trajs 1-10, Chunk 1 = trajs 11-20, etc.
df_filtered['Chunk_Int'] = (df_filtered['Trajectory No.'] - 1) // 10

# Group by Bat Name ONLY (merging days)
grouped = df_filtered.groupby('Bat Name')
bats = list(grouped.groups.keys())
bats.sort()  # Sort alphabetically

num_rows = len(bats)

# Prepare dynamic subplot titles
titles = []
for bat in bats:
    titles.append(f"{bat.capitalize()} - Target Hit %")
    titles.append(f"{bat.capitalize()} - Bif (Raw) %")
    titles.append(f"{bat.capitalize()} - Bif (Deriv) %")

fig = make_subplots(
    rows=num_rows, cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.05,
    vertical_spacing=0.12 # Spaced out to prevent title overlapping
)

# Define consistent colors across all plots
all_targets = sorted(df_filtered['Target Hit'].unique().tolist())
target_colors = {t: px.colors.qualitative.Safe[i % len(px.colors.qualitative.Safe)] for i, t in enumerate(all_targets)}
bif_colors = {0: "#d62728", 1: "#1f77b4", 2: "#2ca02c"}  # Red for 0, Blue for 1, Green for 2

# Keep track of which legend items have been added so we don't duplicate them
added_to_legend = set()

for i, bat in enumerate(bats, start=1):
    subset = grouped.get_group(bat)
    
    # ----------------------------------------------------
    # Calculate Percentages per Chunk (Aggregated across days)
    # ----------------------------------------------------
    # normalize='index' calculates the % across each row (chunk)
    target_xtab = pd.crosstab(subset['Chunk_Int'], subset['Target Hit'], normalize='index') * 100
    bif_raw_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Raw)'], normalize='index') * 100
    bif_deriv_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Deriv)'], normalize='index') * 100
    
    # Ensure all possible columns exist (so lines don't break if 0% happen)
    target_xtab = target_xtab.reindex(columns=all_targets, fill_value=0.0)
    bif_raw_xtab = bif_raw_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    bif_deriv_xtab = bif_deriv_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    
    # Generate ordered x-axis labels based on the integer chunks present for this bat
    x_vals = sorted(subset['Chunk_Int'].unique())
    x_labels = [f"{int(c)*10+1}-{int(c)*10+10}" for c in x_vals]
    
    # Align the crosstab indices perfectly with x_vals
    target_xtab = target_xtab.loc[x_vals]
    bif_raw_xtab = bif_raw_xtab.loc[x_vals]
    bif_deriv_xtab = bif_deriv_xtab.loc[x_vals]
    
    # ----------------------------------------------------
    # 1. Target Hit Line Plot
    # ----------------------------------------------------
    for target in all_targets:
        show_leg = (target not in added_to_legend)
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=target_xtab[target].values,
                mode='lines+markers',
                name=target,
                legendgroup='targets',
                legendgrouptitle_text="Targets",
                line=dict(color=target_colors[target], width=2),
                marker=dict(size=8),
                showlegend=show_leg
            ),
            row=i, col=1
        )
        added_to_legend.add(target)
        
    # ----------------------------------------------------
    # 2. Bifurcation Line Plot (Raw)
    # ----------------------------------------------------
    for bif_count in [0, 1, 2]:
        name_str = f"{bif_count} Bifurcation{'s' if bif_count != 1 else ''}"
        show_leg = (name_str not in added_to_legend)
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_raw_xtab[bif_count].values,
                mode='lines+markers',
                name=name_str,
                legendgroup='bifurcations',
                legendgrouptitle_text="Bifurcations",
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=show_leg
            ),
            row=i, col=2
        )
        added_to_legend.add(name_str)

    # ----------------------------------------------------
    # 3. Bifurcation Line Plot (Deriv)
    # ----------------------------------------------------
    for bif_count in [0, 1, 2]:
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_deriv_xtab[bif_count].values,
                mode='lines+markers',
                name=f"{bif_count} Bifurcation{'s' if bif_count != 1 else ''}",
                legendgroup='bifurcations',
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=False  # Already handled in col 2
            ),
            row=i, col=3
        )
        
    # Clean up axes for all three columns
    for col in [1, 2, 3]:
        fig.update_xaxes(
            title_text="Trajectory Chunk", 
            type='category', 
            categoryorder='array', 
            categoryarray=x_labels, 
            row=i, col=col
        )
        fig.update_yaxes(title_text="% of Trajectories", range=[-5, 105], row=i, col=col)

# ----------------------------------------------------
# Draw thick horizontal lines between different bats
# ----------------------------------------------------
for r in range(1, num_rows):
    y_axis_above = 'yaxis' if r == 1 else f'yaxis{(r-1)*3 + 1}'
    y_axis_below = f'yaxis{r*3 + 1}'
    
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=2)
    )

# Increased baseline pixels per row to maintain space
total_height = max(800, 450 * num_rows)

fig.update_layout(
    title=dict(
        text="Overall Temporal Bias per Bat (Aggregated Across Days, Excl. 06.10.22)",
        x=0.5, font=dict(size=22)
    ),
    height=total_height,
    width=1700,
    template="simple_white",
    hovermode="x unified",
    legend=dict(
        yanchor="top", y=1, xanchor="left", x=1.02,
        bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
    )
)

fig.show()

In [19]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# Ensure Chunk columns exist (in case df was freshly reloaded)
if 'Chunk_Int' not in df.columns:
    df['Chunk_Int'] = (df['Trajectory No.'] - 1) // 10

# Define the 3 subsets to loop through: (data_subset, row_index)
datasets = [
    (df, 1),
    (df[df['Modality'] == 'Vision'], 2),
    (df[df['Modality'] == 'Acoustic'], 3)
]

titles = [
    "All Days - Bif (Raw)", "All Days - Bif (Deriv)",
    "Vision Days - Bif (Raw)", "Vision Days - Bif (Deriv)",
    "Acoustic Days - Bif (Raw)", "Acoustic Days - Bif (Deriv)"
]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.18  # Increased from 0.1 to give titles and labels room to breathe
)

# Consistent colors: Red=0, Blue=1, Green=2
bif_colors = {0: "#d62728", 1: "#1f77b4", 2: "#2ca02c"}
added_to_legend = set()

for subset, row_idx in datasets:
    # Skip if a modality doesn't exist in the loaded data (safety check)
    if subset.empty:
        continue
        
    # ----------------------------------------------------
    # Calculate Percentages per Chunk
    # ----------------------------------------------------
    bif_raw_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Raw)'], normalize='index') * 100
    bif_deriv_xtab = pd.crosstab(subset['Chunk_Int'], subset['No. of Bifurcations (Deriv)'], normalize='index') * 100
    
    # Ensure all possible columns [0, 1, 2] exist
    bif_raw_xtab = bif_raw_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    bif_deriv_xtab = bif_deriv_xtab.reindex(columns=[0, 1, 2], fill_value=0.0)
    
    # Generate ordered x-axis labels based on the integer chunks present in this specific subset
    x_vals = sorted(subset['Chunk_Int'].unique())
    x_labels = [f"{int(c)*10+1}-{int(c)*10+10}" for c in x_vals]
    
    # Ensure the crosstab rows perfectly align with our sorted x_vals
    bif_raw_xtab = bif_raw_xtab.loc[x_vals]
    bif_deriv_xtab = bif_deriv_xtab.loc[x_vals]
    
    for bif_count in [0, 1, 2]:
        name_str = f"{bif_count} Bifurcation{'s' if bif_count != 1 else ''}"
        show_leg = (name_str not in added_to_legend)
        
        # ----------------------------------------------------
        # Column 1: Raw Line Plot
        # ----------------------------------------------------
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_raw_xtab[bif_count].values,
                mode='lines+markers',
                name=name_str,
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=show_leg
            ),
            row=row_idx, col=1
        )
        added_to_legend.add(name_str)
        
        # ----------------------------------------------------
        # Column 2: Deriv Line Plot
        # ----------------------------------------------------
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=bif_deriv_xtab[bif_count].values,
                mode='lines+markers',
                name=name_str,
                line=dict(color=bif_colors[bif_count], width=2),
                marker=dict(size=8),
                showlegend=False
            ),
            row=row_idx, col=2
        )
        
    # Clean up axes for both columns in this row
    for col in [1, 2]:
        # Enforce exact category order so chunks don't sort alphabetically
        fig.update_xaxes(
            title_text="Trajectory Chunk", 
            type='category', 
            categoryorder='array', 
            categoryarray=x_labels, 
            row=row_idx, col=col
        )
        fig.update_yaxes(
            title_text="% of Trajectories", 
            range=[-5, 105], 
            row=row_idx, col=col
        )

# ----------------------------------------------------
# Draw thick horizontal lines between the rows
# ----------------------------------------------------
# We have 3 rows, so we need lines between Row 1 & 2, and Row 2 & 3.
for r in [1, 2]:
    # Calculate Plotly's internal y-axis index for the 1st column of the target rows
    above_idx = (r - 1) * 2 + 1
    below_idx = r * 2 + 1
    
    y_axis_above = 'yaxis' if above_idx == 1 else f'yaxis{above_idx}'
    y_axis_below = f'yaxis{below_idx}'
    
    # Get the domain bounds to find the exact visual midpoint
    y_bottom_above = fig.layout[y_axis_above].domain[0]
    y_top_below = fig.layout[y_axis_below].domain[1]
    y_mid = (y_bottom_above + y_top_below) / 2
    
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="paper",  # Span the entire page width
        y0=y_mid, y1=y_mid, yref="paper",
        line=dict(color="black", width=4)
    )

# ----------------------------------------------------
# Global Layout
# ----------------------------------------------------
fig.update_layout(
    title=dict(
        text="Aggregate Temporal Bias: Bifurcation Detection (Raw vs. Deriv)",
        x=0.5, font=dict(size=22)
    ),
    height=1200,  # Increased from 1000 to maintain plot size with the new spacing
    width=1400,
    template="simple_white",
    hovermode="x unified",
    legend=dict(
        title="Bifurcation Count",
        yanchor="top", y=1, xanchor="left", x=1.02,
        bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
    )
)

fig.show()

In [24]:
import pandas as pd
import numpy as np
import ast
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# ----------------------------------------------------
# 1. Data Preparation
# ----------------------------------------------------
# Filter out the 2-target day
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Safe parser for the angle lists
def parse_angles(val):
    if pd.isna(val):
        return None
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return None
    if isinstance(val, (list, np.ndarray)):
        return val
    return None

# Apply the parser
angle_cols = [
    'Angle to Targets at 1st Bifurcation (Raw)',
    'Angle to Targets at 2nd Bifurcation (Raw)',
    'Angle to Targets at 1st Bifurcation (Deriv)',
    'Angle to Targets at 2nd Bifurcation (Deriv)'
]
for col in angle_cols:
    df_filtered[col] = df_filtered[col].apply(parse_angles)

# ----------------------------------------------------
# 2. Setup Conditions
# ----------------------------------------------------
conditions = [
    {"title": "1 Bif (Raw)", "method": "Raw", "bif_total": 1, "target_col": 'Angle to Targets at 1st Bifurcation (Raw)'},
    {"title": "2 Bifs - 1st (Raw)", "method": "Raw", "bif_total": 2, "target_col": 'Angle to Targets at 1st Bifurcation (Raw)'},
    {"title": "2 Bifs - 2nd (Raw)", "method": "Raw", "bif_total": 2, "target_col": 'Angle to Targets at 2nd Bifurcation (Raw)'},
    {"title": "1 Bif (Deriv)", "method": "Deriv", "bif_total": 1, "target_col": 'Angle to Targets at 1st Bifurcation (Deriv)'},
    {"title": "2 Bifs - 1st (Deriv)", "method": "Deriv", "bif_total": 2, "target_col": 'Angle to Targets at 1st Bifurcation (Deriv)'},
    {"title": "2 Bifs - 2nd (Deriv)", "method": "Deriv", "bif_total": 2, "target_col": 'Angle to Targets at 2nd Bifurcation (Deriv)'}
]

target_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

# ----------------------------------------------------
# 3. Figure Generator Function
# ----------------------------------------------------
def create_page(dataset_name, subset):
    # 6 rows (conditions) x 3 cols (targets)
    titles = []
    for cond in conditions:
        for t_idx in range(3):
            titles.append(f"{cond['title']} | Target {t_idx + 1}")
            
    fig = make_subplots(
        rows=6, cols=3,
        subplot_titles=titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.06
    )
    
    for row_idx, cond in enumerate(conditions, start=1):
        count_col = f"No. of Bifurcations ({cond['method']})"
        mask = subset[count_col] == cond['bif_total']
        valid_rows = subset[mask][cond['target_col']].dropna()
        
        for col_idx in range(1, 4):
            t_idx = col_idx - 1
            angles = [val[t_idx] for val in valid_rows if len(val) > t_idx]
            
            if angles:
                fig.add_trace(
                    go.Histogram(
                        x=angles,
                        name=f"Target {t_idx + 1}",
                        marker_color=target_colors[t_idx],
                        xbins=dict(start=0, end=180, size=5),
                        showlegend=False
                    ),
                    row=row_idx, col=col_idx
                )
                
                # Calculate and draw the mean line
                mean_val = np.mean(angles)
                fig.add_vline(
                    x=mean_val, 
                    line_dash="dash", 
                    line_color="black", 
                    line_width=2,
                    annotation_text=f"Mean: {mean_val:.1f}°", 
                    annotation_position="top right",
                    row=row_idx, col=col_idx
                )
                
            fig.update_xaxes(title_text="Angle (°)", range=[0, 180], row=row_idx, col=col_idx)
            if col_idx == 1:
                fig.update_yaxes(title_text="Count", row=row_idx, col=col_idx)

    # Add thick horizontal dividers between the rows for readability
    for r in range(1, 6):
        above_idx = (r - 1) * 3 + 1
        below_idx = r * 3 + 1
        
        y_axis_above = 'yaxis' if above_idx == 1 else f'yaxis{above_idx}'
        y_axis_below = f'yaxis{below_idx}'
        
        y_bottom_above = fig.layout[y_axis_above].domain[0]
        y_top_below = fig.layout[y_axis_below].domain[1]
        y_mid = (y_bottom_above + y_top_below) / 2
        
        fig.add_shape(
            type="line",
            x0=0, x1=1, xref="paper",
            y0=y_mid, y1=y_mid, yref="paper",
            line=dict(color="black", width=3)
        )

    fig.update_layout(
        title=dict(
            text=f"Distribution of Critical Angles: {dataset_name} (Excl. 06.10.22)",
            x=0.5, font=dict(size=22)
        ),
        height=1800,
        width=1400,
        template="simple_white"
    )
    
    # Opens this specific figure in a new browser tab
    fig.show()

# ----------------------------------------------------
# 4. Generate the 3 Separate Pages
# ----------------------------------------------------
# This will open 3 separate tabs in your default browser
create_page("All Days", df_filtered)
create_page("Vision Days", df_filtered[df_filtered['Modality'] == 'Vision'])
create_page("Acoustic Days", df_filtered[df_filtered['Modality'] == 'Acoustic'])

In [24]:
import pandas as pd
import numpy as np
import ast
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "browser"

# ----------------------------------------------------
# 1. Data Preparation
# ----------------------------------------------------
# Filter out the 2-target day
df_filtered = df[df['Date'] != '06.10.22'].copy()

# Standardize bat names to lowercase (if not already done)
df_filtered['Bat Name'] = df_filtered['Bat Name'].astype(str).str.lower()

# Safe parser for the angle lists
def parse_angles(val):
    if pd.isna(val):
        return None
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return None
    if isinstance(val, (list, np.ndarray)):
        return val
    return None

# Apply the parser
angle_cols = [
    'Angle to Targets at 1st Bifurcation (Raw)',
    'Angle to Targets at 2nd Bifurcation (Raw)',
    'Angle to Targets at 1st Bifurcation (Deriv)',
    'Angle to Targets at 2nd Bifurcation (Deriv)'
]
for col in angle_cols:
    df_filtered[col] = df_filtered[col].apply(parse_angles)

# ----------------------------------------------------
# 2. Setup Conditions
# ----------------------------------------------------
conditions = [
    {"title": "1 Bif (Raw)", "method": "Raw", "bif_total": 1, "target_col": 'Angle to Targets at 1st Bifurcation (Raw)'},
    {"title": "2 Bifs - 1st (Raw)", "method": "Raw", "bif_total": 2, "target_col": 'Angle to Targets at 1st Bifurcation (Raw)'},
    {"title": "2 Bifs - 2nd (Raw)", "method": "Raw", "bif_total": 2, "target_col": 'Angle to Targets at 2nd Bifurcation (Raw)'},
    {"title": "1 Bif (Deriv)", "method": "Deriv", "bif_total": 1, "target_col": 'Angle to Targets at 1st Bifurcation (Deriv)'},
    {"title": "2 Bifs - 1st (Deriv)", "method": "Deriv", "bif_total": 2, "target_col": 'Angle to Targets at 1st Bifurcation (Deriv)'},
    {"title": "2 Bifs - 2nd (Deriv)", "method": "Deriv", "bif_total": 2, "target_col": 'Angle to Targets at 2nd Bifurcation (Deriv)'}
]

target_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

# ----------------------------------------------------
# 3. Figure Generator Function
# ----------------------------------------------------
def create_page(dataset_name, subset):
    # Safety check: If a bat didn't fly in a specific modality, don't generate a blank page
    if subset.empty:
        return
        
    # 6 rows (conditions) x 3 cols (targets)
    titles = []
    for cond in conditions:
        for t_idx in range(3):
            titles.append(f"{cond['title']} | Target {t_idx + 1}")
            
    fig = make_subplots(
        rows=6, cols=3,
        subplot_titles=titles,
        horizontal_spacing=0.05,
        vertical_spacing=0.06
    )
    
    for row_idx, cond in enumerate(conditions, start=1):
        count_col = f"No. of Bifurcations ({cond['method']})"
        mask = subset[count_col] == cond['bif_total']
        valid_rows = subset[mask][cond['target_col']].dropna()
        
        for col_idx in range(1, 4):
            t_idx = col_idx - 1
            angles = [val[t_idx] for val in valid_rows if len(val) > t_idx]
            
            if angles:
                fig.add_trace(
                    go.Histogram(
                        x=angles,
                        name=f"Target {t_idx + 1}",
                        marker_color=target_colors[t_idx],
                        xbins=dict(start=0, end=180, size=5),
                        showlegend=False
                    ),
                    row=row_idx, col=col_idx
                )
                
                # Calculate and draw the mean line
                mean_val = np.mean(angles)
                fig.add_vline(
                    x=mean_val, 
                    line_dash="dash", 
                    line_color="black", 
                    line_width=2,
                    annotation_text=f"Mean: {mean_val:.1f}°", 
                    annotation_position="top right",
                    row=row_idx, col=col_idx
                )
                
            fig.update_xaxes(title_text="Angle (°)", range=[0, 180], row=row_idx, col=col_idx)
            if col_idx == 1:
                fig.update_yaxes(title_text="Count", row=row_idx, col=col_idx)

    # Add thick horizontal dividers between the rows for readability
    for r in range(1, 6):
        above_idx = (r - 1) * 3 + 1
        below_idx = r * 3 + 1
        
        y_axis_above = 'yaxis' if above_idx == 1 else f'yaxis{above_idx}'
        y_axis_below = f'yaxis{below_idx}'
        
        y_bottom_above = fig.layout[y_axis_above].domain[0]
        y_top_below = fig.layout[y_axis_below].domain[1]
        y_mid = (y_bottom_above + y_top_below) / 2
        
        fig.add_shape(
            type="line",
            x0=0, x1=1, xref="paper",
            y0=y_mid, y1=y_mid, yref="paper",
            line=dict(color="black", width=3)
        )

    fig.update_layout(
        title=dict(
            text=f"Distribution of Critical Angles: {dataset_name} (Excl. 06.10.22)",
            x=0.5, font=dict(size=22)
        ),
        height=1800,
        width=1400,
        template="simple_white"
    )
    
    # Opens this specific figure in a new browser tab
    fig.show()

# ----------------------------------------------------
# 4. Generate the Separate Pages Per Bat
# ----------------------------------------------------
# Get unique bats and sort them alphabetically
unique_bats = sorted(df_filtered['Bat Name'].dropna().unique())

for bat in unique_bats:
    # Subset for the specific bat
    bat_df = df_filtered[df_filtered['Bat Name'] == bat]
    
    # 1. All Days for this bat
    create_page(f"All Days | {bat.capitalize()}", bat_df)
    
    # 2. Vision Days for this bat
    vision_df = bat_df[bat_df['Modality'] == 'Vision']
    create_page(f"Vision Days | {bat.capitalize()}", vision_df)
    
    # 3. Acoustic Days for this bat
    acoustic_df = bat_df[bat_df['Modality'] == 'Acoustic']
    create_page(f"Acoustic Days | {bat.capitalize()}", acoustic_df)

## 6. Multi-Trajectory Analysis

Runs bifurcation detection across a set of trajectories, clusters the resulting bifurcation points with DBSCAN to get their center(s) of mass, and classifies/extrapolates each trajectory to determine which target (if any) it lands on, plotting both the full 3D picture and the landing footprint on each target.

In [5]:
from sklearn.cluster import DBSCAN
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter

pio.renderers.default = "browser"

# =====================================================
# 3D GEOMETRY HELPER FUNCTIONS
# =====================================================

def get_hitbox_params(rect, depth=300, side_padding=0):
    p0, p1, p2, p3 = rect[0], rect[1], rect[2], rect[3]
    u = p1 - p0
    v = p3 - p0
    L_u = np.linalg.norm(u) + (side_padding * 2)
    L_v = np.linalg.norm(v) + (side_padding * 2)
    u_hat = u / np.linalg.norm(u)
    v_hat = v / np.linalg.norm(v)
    n_hat = np.cross(u_hat, v_hat)
    center = (p0 + p2) / 2.0
    return {"center": center, "u": u_hat, "L_u": L_u,
            "v": v_hat, "L_v": L_v, "n": n_hat, "L_n": depth}

def is_point_in_hitbox(pt, box):
    v = pt - box["center"]
    if abs(np.dot(v, box["u"])) > box["L_u"] / 2.0: return False
    if abs(np.dot(v, box["v"])) > box["L_v"] / 2.0: return False
    if abs(np.dot(v, box["n"])) > box["L_n"] / 2.0: return False
    return True

RECT_COLORS = ["black", "magenta", "cyan", "orange", "purple"]

# =====================================================
# MAIN FUNCTION
# =====================================================

def analyze_target_trajectories_plotly(
    selected_date,
    selected_bat_trajectories,
    required_hold_steps=50,
    window_size=25,
    min_corr_threshold=0.8,
    plateau_hold_steps=1000,
    next_start=500,
    trajectory_fraction=0.85,
    # Smoothing parameters for derivative method
    angle_smoothing_window=51,
    angle_smoothing_polyorder=3,
    time_column="Time",
    # Clustering parameters
    cluster_eps=100.0,
    cluster_min_samples=10,
    eps_3=None, min_samples_3=None,
    eps_2=None, min_samples_2=None,
    eps_1=None, min_samples_1=None,
    hitbox_depth=200,
    hitbox_padding=50,
    extrapolation_step=10,
    max_extrapolation=200,
    start_skip=10
):
    # Helper to parse eps/min_samples for a given number of orders
    def _get_params(specific_val, global_val, n_orders):
        if specific_val is not None:
            val = specific_val
        else:
            val = global_val
        if isinstance(val, (int, float)):
            return [val] * n_orders
        else:
            return list(val)[:n_orders]

    # =====================================================
    # LOAD MASTER FILES (LOW & HIGH)
    # =====================================================
    df_low = pd.read_csv(MASTER_FILES_LOW[selected_date])
    df_low["trajectory_no"] = pd.to_numeric(df_low["trajectory_no"], errors="coerce")
    
    df_high = pd.read_csv(MASTER_FILES_HIGH[selected_date])
    df_high["trajectory_no"] = pd.to_numeric(df_high["trajectory_no"], errors="coerce")

    rects = get_rectangles(selected_date)
    target_centers = np.array([rectangle_center(rect) for rect in rects])

    print("\n" + "="*50)
    print("TARGET CENTERS")
    for i, target in enumerate(target_centers, start=1):
        print(f"Target {i}: {target}")

    # Build hitboxes
    hitbox_targets = [
        {
            "name": f"Rectangle {i+1}",
            "rect": rect,
            "color": RECT_COLORS[i % len(RECT_COLORS)],
            "box": get_hitbox_params(rect, hitbox_depth, hitbox_padding)
        }
        for i, rect in enumerate(rects)
    ]

    # =====================================================
    # ANALYZE TRAJECTORIES
    # =====================================================
    all_results_combined = {}
    start_points = []
    
    # Initialize trackers for BOTH methods
    def init_traj_groups():
        return {
            3: {"results": {}, "bif_by_order": {0: [], 1: [], 2: []}},
            2: {"results": {}, "bif_by_order": {0: [], 1: []}},
            1: {"results": {}, "bif_by_order": {0: []}},
            0: {"results": {}, "bif_by_order": {}},
        }
    
    traj_groups_raw = init_traj_groups()
    traj_groups_deriv = init_traj_groups()
    
    landing_points = {ht["name"]: [] for ht in hitbox_targets}

    for bat_name, traj_list in selected_bat_trajectories.items():
        for traj_no in traj_list:
            
            # Fetch both low and high traces
            mask_low = (df_low["bat_name"].astype(str) == bat_name) & (df_low["trajectory_no"] == traj_no)
            traj_low = df_low[mask_low].dropna(subset=["pos_x", "pos_y", "pos_z"])
            
            mask_high = (df_high["bat_name"].astype(str) == bat_name) & (df_high["trajectory_no"] == traj_no)
            traj_high = df_high[mask_high].dropna(subset=["pos_x", "pos_y", "pos_z"])

            if traj_low.empty or traj_high.empty:
                continue
                
            xs_low = traj_low[["pos_x", "pos_y", "pos_z"]].values[start_skip:]
            xs_high = traj_high[["pos_x", "pos_y", "pos_z"]].values[start_skip:]
            
            if len(xs_low) < 2 or len(xs_high) < 2:
                continue

            if time_column in traj_high.columns:
                time_full = traj_high[time_column].values
            else:
                time_full = np.arange(len(traj_high))

            # ----- Landing classification (on xs_high) -----
            hit_target = None; hit_target_idx = None; hit_pt = None; hit_idx = None
            extrapolated_pts = []
            for idx_pt, pt in enumerate(xs_high):
                for idx, ht in enumerate(hitbox_targets):
                    if is_point_in_hitbox(pt, ht["box"]):
                        hit_target = ht; hit_target_idx = idx; hit_pt = pt.copy(); hit_idx = idx_pt
                        break
                if hit_target:
                    break
            
            if not hit_target:
                tail_len = max(2, int(len(xs_high) * 0.1))
                velocity = xs_high[-1] - xs_high[-tail_len]
                speed = np.linalg.norm(velocity)
                if speed > 0:
                    direction = velocity / speed
                    curr_pt = xs_high[-1].copy()
                    for _ in range(max_extrapolation):
                        curr_pt = curr_pt + direction * extrapolation_step
                        extrapolated_pts.append(curr_pt.copy())
                        for idx, ht in enumerate(hitbox_targets):
                            if is_point_in_hitbox(curr_pt, ht["box"]):
                                hit_target = ht; hit_target_idx = idx; hit_pt = curr_pt.copy()
                                break
                        if hit_target:
                            break

            # Ignore trajectories that never hit any target or only hit during extrapolation
            if hit_target is None or hit_idx is None:
                continue

            start_points.append(xs_high[0])

            landing_point = None
            if hit_target and hit_pt is not None:
                box = hit_target["box"]; c = box["center"]; n = box["n"]
                v = hit_pt - c
                dist_to_plane = np.dot(v, n)
                landing_point = hit_pt - (dist_to_plane * n)
                landing_points[hit_target["name"]].append({"pt": landing_point, "bat": bat_name, "traj": traj_no})

            end_pt = hit_pt if hit_pt is not None else (extrapolated_pts[-1] if extrapolated_pts else xs_high[-1])

            # Truncate BOTH trajectories at landing
            xs_bif_low = xs_low[:hit_idx+1]
            xs_bif_high = xs_high[:hit_idx+1]

            if len(xs_bif_high) < 2:
                continue

            bifurcation_targets = target_centers.copy()
            if hit_target is not None and landing_point is not None:
                bifurcation_targets[hit_target_idx] = landing_point

            # ===============================================
            # 1. RAW DETECTION (low‑smoothed)
            # ===============================================
            results_raw = find_bifurcations(
                xs_bif_low, bifurcation_targets,
                required_hold_steps=required_hold_steps,
                window_size=window_size,
                min_corr_threshold=min_corr_threshold,
                plateau_hold_steps=plateau_hold_steps,
                next_start=next_start,
                trajectory_fraction=trajectory_fraction
            )

            # ===============================================
            # 2. DERIVATIVE DETECTION (high‑smoothed)
            # ===============================================
            vel_high = np.diff(xs_bif_high, axis=0)
            vn_high = np.linalg.norm(vel_high, axis=1, keepdims=True)
            vu_high = vel_high / (vn_high + 1e-12)
            pos_high = xs_bif_high[:-1]

            n_targ = len(hitbox_targets)
            angles_high = np.zeros((len(pos_high), n_targ))
            for i, t in enumerate(target_centers):
                d = t - pos_high
                dn = np.linalg.norm(d, axis=1, keepdims=True)
                du = d / (dn + 1e-12)
                dots = np.sum(vu_high * du, axis=1)
                dots = np.clip(dots, -1, 1)
                angles_high[:, i] = np.degrees(np.arccos(dots))

            t_bif_high = time_full[:len(xs_bif_high)]
            t_axis_high = t_bif_high[:-1]

            data_len = len(angles_high)
            win = min(angle_smoothing_window, data_len - (1 if data_len % 2 == 0 else 0))
            if win <= angle_smoothing_polyorder:
                win = data_len - (1 if data_len % 2 == 0 else 0)
                poly = win - 1 if win > 1 else 0
            else:
                poly = angle_smoothing_polyorder

            if win > 2 and poly > 0:
                smooth_high = savgol_filter(angles_high, window_length=win, polyorder=poly, axis=0)
            else:
                smooth_high = angles_high

            d1_high = np.gradient(smooth_high, t_axis_high, axis=0)
            if win > 2 and poly > 0:
                d1_smooth = savgol_filter(d1_high, window_length=win, polyorder=poly, axis=0)
            else:
                d1_smooth = d1_high

            results_deriv = find_bifurcations(
                xs_bif_high, bifurcation_targets,
                required_hold_steps=required_hold_steps,
                window_size=window_size,
                min_corr_threshold=min_corr_threshold,
                plateau_hold_steps=plateau_hold_steps,
                next_start=next_start,
                trajectory_fraction=trajectory_fraction,
                angles_deg=d1_smooth
            )

            # Store overall shared payload
            payload = {
                "xs_high": xs_high, "xs_bif_high": xs_bif_high,
                "xs_low": xs_low, "xs_bif_low": xs_bif_low,
                "hit_target": hit_target, "end_pt": end_pt
            }
            all_results_combined[(bat_name, traj_no)] = payload

            # Categorize Raw
            n_bif_raw = min(len(results_raw["bif_indices"]), 3)
            group_raw = traj_groups_raw.get(n_bif_raw)
            if group_raw is not None:
                group_raw["results"][(bat_name, traj_no)] = payload
                for order, idx in enumerate(results_raw["bif_indices"][:n_bif_raw]):
                    group_raw["bif_by_order"][order].append(results_raw["positions"][idx])

            # Categorize Deriv
            n_bif_deriv = min(len(results_deriv["bif_indices"]), 3)
            group_deriv = traj_groups_deriv.get(n_bif_deriv)
            if group_deriv is not None:
                group_deriv["results"][(bat_name, traj_no)] = payload
                for order, idx in enumerate(results_deriv["bif_indices"][:n_bif_deriv]):
                    group_deriv["bif_by_order"][order].append(results_deriv["positions"][idx])

    if len(all_results_combined) == 0:
        raise ValueError("No valid trajectories found.")

    avg_start = np.mean(start_points, axis=0)
    print("\nAverage Start Position:")
    print(avg_start)

    # =====================================================
    # DBSCAN CLUSTERING PER METHOD, GROUP & ORDER
    # =====================================================
    order_names = {3: ["1st Bifurcation", "2nd Bifurcation", "3rd Bifurcation"],
                   2: ["1st Bifurcation", "2nd Bifurcation"],
                   1: ["1st Bifurcation"]}

    def perform_clustering(traj_groups):
        cluster_data = {}
        for n_bif, group in traj_groups.items():
            if n_bif == 0: continue
            centers_dict = {}
            points_dict = {}
            if n_bif == 3:
                eps_list = _get_params(eps_3, cluster_eps, 3)
                min_list = _get_params(min_samples_3, cluster_min_samples, 3)
            elif n_bif == 2:
                eps_list = _get_params(eps_2, cluster_eps, 2)
                min_list = _get_params(min_samples_2, cluster_min_samples, 2)
            elif n_bif == 1:
                eps_list = _get_params(eps_1, cluster_eps, 1)
                min_list = _get_params(min_samples_1, cluster_min_samples, 1)

            for order in group["bif_by_order"]:
                pts = group["bif_by_order"][order]
                if len(pts) == 0:
                    centers_dict[order] = []
                    points_dict[order] = {}
                    continue
                pts_arr = np.array(pts)
                clustering = DBSCAN(eps=eps_list[order], min_samples=min_list[order]).fit(pts_arr)
                labels = clustering.labels_
                centers = []
                pdict = {}
                for lbl in set(labels):
                    if lbl == -1:
                        pdict[-1] = pts_arr[labels == -1]
                    else:
                        mask = labels == lbl
                        cluster_pts = pts_arr[mask]
                        com = cluster_pts.mean(axis=0)
                        centers.append(com)
                        pdict[lbl] = cluster_pts
                centers_dict[order] = centers
                points_dict[order] = pdict
            cluster_data[n_bif] = {"centers": centers_dict, "points": points_dict}
        return cluster_data

    cluster_data_raw = perform_clustering(traj_groups_raw)
    cluster_data_deriv = perform_clustering(traj_groups_deriv)

    # =====================================================
    # FIGURE 1: ALL TRAJECTORIES
    # =====================================================
    fig1 = go.Figure()
    for ht in hitbox_targets:
        fig1.add_trace(go.Scatter3d(
            x=ht["rect"][:,0], y=ht["rect"][:,1], z=ht["rect"][:,2],
            mode="lines", line=dict(color=ht["color"], width=8), name=ht["name"]))
        box = ht["box"]; c = box["center"]
        u_vec = box["u"] * (box["L_u"]/2); v_vec = box["v"] * (box["L_v"]/2); n_vec = box["n"] * (box["L_n"]/2)
        p0 = c - u_vec - v_vec; p1 = c + u_vec - v_vec; p2 = c + u_vec + v_vec; p3 = c - u_vec + v_vec
        padded_face = np.array([p0, p1, p2, p3])
        bottom_face = padded_face - n_vec; top_face = padded_face + n_vec
        b_loop = np.vstack([bottom_face, bottom_face[0]])
        t_loop = np.vstack([top_face, top_face[0]])
        def add_box_edge(loop, col):
            fig1.add_trace(go.Scatter3d(x=loop[:,0], y=loop[:,1], z=loop[:,2],
                            mode="lines", line=dict(color=col, width=2, dash="dash"),
                            opacity=0.3, showlegend=False))
        add_box_edge(b_loop, ht["color"]); add_box_edge(t_loop, ht["color"])
        for j in range(4):
            edge = np.vstack([bottom_face[j], top_face[j]])
            add_box_edge(edge, ht["color"])

    fig1.add_trace(go.Scatter3d(x=target_centers[:,0], y=target_centers[:,1], z=target_centers[:,2],
                    mode="markers", marker=dict(color="red", size=7), name="Target Centers"))

    for key, data in all_results_combined.items():
        xs = data["xs_high"]
        hit_target = data["hit_target"]
        end_pt = data["end_pt"]
        color = hit_target["color"] if hit_target else "gray"
        fig1.add_trace(go.Scatter3d(x=xs[:,0], y=xs[:,1], z=xs[:,2],
                        mode="lines", line=dict(color=color, width=5), opacity=0.8,
                        name=f"{key[0]}-{key[1]}"))
        fig1.add_trace(go.Scatter3d(x=[xs[0,0]], y=[xs[0,1]], z=[xs[0,2]],
                        mode="markers", marker=dict(size=4, color=color, symbol="circle"), showlegend=False))
        fig1.add_trace(go.Scatter3d(x=[end_pt[0]], y=[end_pt[1]], z=[end_pt[2]],
                        mode="markers", marker=dict(size=5, color="red", symbol="square"), showlegend=False))

    fig1.add_trace(go.Scatter3d(x=[avg_start[0]], y=[avg_start[1]], z=[avg_start[2]],
                    mode="markers+text", marker=dict(color="lime", size=10),
                    text=["Avg Start"], textposition="top center", name="Average Start"))
    fig1.update_layout(title=f"{selected_date} | All {len(all_results_combined)} trajectories",
                       width=1200, height=900,
                       scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
                                  aspectmode="manual", aspectratio=dict(x=1, y=1, z=1)),
                       legend=dict(yanchor="top", y=1, xanchor="left", x=1.02))
    fig1.show()

    # =====================================================
    # GENERATE GROUP FIGURES FOR BOTH METHODS
    # =====================================================
    cluster_colors = ["blue", "green", "orange", "purple", "brown", "pink", "gray", "olive"]

    methods_to_plot = [
        ("Raw", traj_groups_raw, cluster_data_raw, cluster_data_deriv, "xs_low"),
        ("Deriv", traj_groups_deriv, cluster_data_deriv, cluster_data_raw, "xs_high")
    ]

    for method_name, t_groups, c_data, c_data_overlay, xs_key in methods_to_plot:
        for n_bif in [3, 2, 1, 0]:
            group = t_groups.get(n_bif)
            if not group or len(group["results"]) == 0:
                continue
            
            fig = go.Figure()
            
            # Hitboxes
            for ht in hitbox_targets:
                fig.add_trace(go.Scatter3d(
                    x=ht["rect"][:,0], y=ht["rect"][:,1], z=ht["rect"][:,2],
                    mode="lines", line=dict(color=ht["color"], width=4), name=ht["name"], opacity=0.6))

            # Trajectories for this method/group
            for key, data in group["results"].items():
                xs = data[xs_key]
                hit_target = data["hit_target"]
                end_pt = data["end_pt"]
                color = hit_target["color"] if hit_target else "gray"
                fig.add_trace(go.Scatter3d(x=xs[:,0], y=xs[:,1], z=xs[:,2],
                                mode="lines", line=dict(color=color, width=3), opacity=0.4,
                                name=f"{key[0]}-{key[1]}"))
                fig.add_trace(go.Scatter3d(x=[xs[0,0]], y=[xs[0,1]], z=[xs[0,2]],
                                mode="markers", marker=dict(size=3, color=color, symbol="circle"), showlegend=False))
                fig.add_trace(go.Scatter3d(x=[end_pt[0]], y=[end_pt[1]], z=[end_pt[2]],
                                mode="markers", marker=dict(size=5, color="red", symbol="square"), showlegend=False))

            # Primary Clusters & COMs
            if n_bif > 0:
                cd = c_data[n_bif]
                for order in group["bif_by_order"]:
                    points_dict = cd["points"][order]
                    centers = cd["centers"][order]
                    cluster_idx = 0
                    for label, pts_arr in points_dict.items():
                        if label == -1:
                            fig.add_trace(go.Scatter3d(x=pts_arr[:,0], y=pts_arr[:,1], z=pts_arr[:,2],
                                            mode="markers", marker=dict(size=3, color="lightgrey", opacity=0.5),
                                            name=f"{order_names[n_bif][order]} noise", showlegend=(order==0)))
                            continue
                        col = cluster_colors[cluster_idx % len(cluster_colors)]
                        cluster_idx += 1
                        fig.add_trace(go.Scatter3d(x=pts_arr[:,0], y=pts_arr[:,1], z=pts_arr[:,2],
                                        mode="markers", marker=dict(size=5, color=col, opacity=0.8),
                                        name=f"{order_names[n_bif][order]} cluster {label}"))
                    
                    for i, com in enumerate(centers):
                        col = cluster_colors[i % len(cluster_colors)]
                        fig.add_trace(go.Scatter3d(x=[com[0]], y=[com[1]], z=[com[2]],
                                        mode="markers+text",
                                        marker=dict(size=10, color=col, symbol="diamond", line=dict(color="black", width=1)),
                                        text=[f"{order_names[n_bif][order]} {method_name} COM"],
                                        textposition="top center",
                                        name=f"{order_names[n_bif][order]} {method_name} COM {i+1}"))

                # OVERLAY COMs from the opposite method (Distinct white square with thick black border)
                overlay_cd = c_data_overlay.get(n_bif)
                if overlay_cd:
                    overlay_name = "Deriv" if method_name == "Raw" else "Raw"
                    for order in group["bif_by_order"]:
                        if order in overlay_cd["centers"]:
                            overlay_centers = overlay_cd["centers"][order]
                            for i, com in enumerate(overlay_centers):
                                fig.add_trace(go.Scatter3d(
                                    x=[com[0]], y=[com[1]], z=[com[2]],
                                    mode="markers+text",
                                    marker=dict(size=12, color="white", symbol="square", line=dict(color="black", width=3)),
                                    text=[f"{overlay_name} COM (Overlay)"],
                                    textposition="bottom center",
                                    name=f"Overlay {overlay_name} COM {order+1}"
                                ))

            title_str = f"[{method_name} Method] {selected_date} | {len(group['results'])} trajectories with {n_bif} bifurcation(s)"
            fig.update_layout(
                title=title_str,
                width=1200, height=900,
                scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
                           aspectmode="manual", aspectratio=dict(x=1, y=1, z=1)),
                legend=dict(yanchor="top", y=1, xanchor="left", x=1.02))
            fig.show()

    # =====================================================
    # FIGURE: LANDING FOOTPRINTS (unchanged)
    # =====================================================
    fig5 = go.Figure()
    for ht in hitbox_targets:
        fig5.add_trace(go.Scatter3d(x=ht["rect"][:,0], y=ht["rect"][:,1], z=ht["rect"][:,2],
                        mode="lines", line=dict(color=ht["color"], width=6),
                        name=f"{ht['name']} Physical Bounds"))
        box = ht["box"]; c = box["center"]
        u_vec = box["u"] * (box["L_u"]/2); v_vec = box["v"] * (box["L_v"]/2)
        p0 = c - u_vec - v_vec; p1 = c + u_vec - v_vec; p2 = c + u_vec + v_vec; p3 = c - u_vec + v_vec
        padded_face = np.array([p0, p1, p2, p3, p0])
        fig5.add_trace(go.Scatter3d(x=padded_face[:,0], y=padded_face[:,1], z=padded_face[:,2],
                        mode="lines", line=dict(color=ht["color"], width=2, dash="dot"),
                        opacity=0.6, name=f"{ht['name']} Capture Area"))
        pts = landing_points[ht["name"]]
        if pts:
            pts_arr = np.array([p["pt"] for p in pts])
            texts = [f"Bat: {p['bat']}<br>Traj: {p['traj']}" for p in pts]
            fig5.add_trace(go.Scatter3d(x=pts_arr[:,0], y=pts_arr[:,1], z=pts_arr[:,2],
                            mode="markers", marker=dict(color=ht["color"], size=8, symbol="circle",
                                                        line=dict(color="black", width=1)),
                            text=texts, hoverinfo="text+name",
                            name=f"{ht['name']} Landing Points"))
    fig5.update_layout(title=f"Landing Footprints | {selected_date}",
                       width=1000, height=800,
                       scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
                       legend=dict(yanchor="top", y=1, xanchor="left", x=1.02))
    fig5.show()

    return {
        "all_results_combined": all_results_combined,
        "traj_groups_raw": traj_groups_raw,
        "traj_groups_deriv": traj_groups_deriv,
        "cluster_data_raw": cluster_data_raw,
        "cluster_data_deriv": cluster_data_deriv,
        "avg_start": avg_start,
        "landing_points": landing_points
    }

In [6]:
# =====================================================
# EXECUTION
# =====================================================

selected_date = "26.12.24"

selected_bat_trajectories = {
    "ketem":list(range(1, 73)), #26 Dec
    "motek":list(range(1, 52)), #26 Dec
    "pitt":list(range(1, 62)) #26 Dec
}

results = analyze_target_trajectories_plotly(
    selected_date,
    selected_bat_trajectories,
    required_hold_steps=30,
    window_size=25,
    min_corr_threshold=0.7,      # NEW – plateau detection (default: 0.8)
    plateau_hold_steps=18,     # NEW – how long min_corr must stay high
    next_start=0,              # NEW – guard after first standard bifurcation
    eps_3=[170, 170, 170], min_samples_3=[5, 5, 5],
    eps_2=[170, 170],       min_samples_2=[5, 5],
    eps_1=[170],             min_samples_1=[5]
)


TARGET CENTERS
Target 1: [ 1092.9094225 -1196.3132325  1038.5306425]
Target 2: [ -605.59549   -1228.9601725  1065.8761   ]
Target 3: [  266.6807125 -1267.2450275  1070.607405 ]


C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_23808\1553168555.py:150: RuntimeWarnin


Average Start Position:
[ 269.41452193 1868.37209338  671.82201611]


In [ ]:
    #"ketem":list(range(1, 33)), #3 Nov (Vision)
    #"pitt":list(range(1, 48)) #3 Nov (Vision)
    #"ketem":list(range(1, 1012)), #01 Oct (Vision)
    #"motek":list(range(1, 38)) #27 Oct (Vision)
    #"motek":list(range(1, 98)) #06 Oct (Vision)
    #"pitt":list(range(1, 62)) #24 Dec
    #"ketem":list(range(1, 73)), #26 Dec
    #"motek":list(range(1, 52)), #26 Dec
    #"pitt":list(range(1, 62)) #26 Dec
    #"brad":list(range(1, 36)), #30 Oct
    #"ketem":list(range(1, 16)), #30 Oct
    #"Pitt":list(range(1, 52)), #30 Oct
    #"ketem":list(range(1, 62)), #31 Dec

## Statistics 

In [5]:
SELECTED_TRAJECTORIES = {
    "30.10.24": {
        "brad": list(range(1, 36)),
        "ketem": list(range(1, 16)),
        "Pitt": list(range(1, 52)),
    },
    "24.12.24": {
        "pitt": list(range(1, 62)),
    },
    "26.12.24": {
        "ketem": list(range(1, 73)),
        "motek": list(range(1, 52)),
        "pitt": list(range(1, 62)),
    },
    "31.12.24": {
        "ketem": list(range(1, 62)),
    },
    "03.11.22": {
        "ketem": list(range(1, 33)),
        "pitt": list(range(1, 48)),
    },
    "01.10.22": {
        "ketem": list(range(1, 1012)),
    },
    "27.10.22": {
        "motek": list(range(1, 38)),
    },
    # "06.10.22" deliberately omitted
}

In [9]:
import pandas as pd
import numpy as np

# ============================================================
# Parameters (same as multi‑trajectory)
# ============================================================
REQUIRED_HOLD_STEPS = 25
WINDOW_SIZE = 25
MIN_CORR_THRESHOLD = 0.7
PLATEAU_HOLD_STEPS = 18
NEXT_START = 0
TRAJECTORY_FRACTION = 0.7          # match multi‑trajectory code

# ============================================================
# Helper: initial max-angle target, using landing-point substitution
# ============================================================
def initial_max_angle_target_with_landing(
    xs_bif, targets_centers, hit_idx_in_xs_bif, hit_target_idx, hitbox_targets, fraction=0.1
):
    N = len(xs_bif)
    n_initial = max(2, int(N * fraction))
    seg = xs_bif[:n_initial]

    # Build modified target positions (landing point for the hit target)
    modified_targets = targets_centers.copy()
    hit_pt = xs_bif[hit_idx_in_xs_bif]
    box = hitbox_targets[hit_target_idx]["box"]
    c = box["center"]
    n_vec = box["n"]
    v = hit_pt - c
    dist_to_plane = np.dot(v, n_vec)
    landing_point = hit_pt - dist_to_plane * n_vec
    modified_targets[hit_target_idx] = landing_point

    # Angles over the initial segment
    vel = np.diff(seg, axis=0)
    vel_norms = np.linalg.norm(vel, axis=1, keepdims=True)
    vel_units = vel / (vel_norms + 1e-12)
    positions = seg[:-1]

    angles = np.zeros((len(positions), len(modified_targets)))
    for i, t in enumerate(modified_targets):
        dir_to_target = t - positions
        dir_norms = np.linalg.norm(dir_to_target, axis=1, keepdims=True)
        dir_units = dir_to_target / (dir_norms + 1e-12)
        dots = np.sum(vel_units * dir_units, axis=1)
        dots = np.clip(dots, -1.0, 1.0)
        angles[:, i] = np.degrees(np.arccos(dots))

    mean_angles = angles.mean(axis=0)
    return np.argmax(mean_angles)

# ============================================================
# Counters
# ============================================================
total_valid = 0
match_total = 0
counts = {0: [0, 0], 1: [0, 0], 2: [0, 0]}   # [total, match] for each n_bif

for date, bat_traj_dict in SELECTED_TRAJECTORIES.items():
    print(f"Processing {date} ...")
    df = pd.read_csv(MASTER_FILES[date])
    df["trajectory_no"] = pd.to_numeric(df["trajectory_no"], errors="coerce")

    rects = get_rectangles(date)
    target_centers = np.array([rectangle_center(rect) for rect in rects])

    # Build hitboxes (needed for landing point)
    hitbox_targets = [{"rect": rect, "box": get_hitbox_params(rect, 200, 50)} for rect in rects]

    for bat, traj_list in bat_traj_dict.items():
        for traj_no in traj_list:
            mask = (df["bat_name"].astype(str) == bat) & (df["trajectory_no"] == traj_no)
            traj_df = df[mask].dropna(subset=["pos_x", "pos_y", "pos_z"])
            if traj_df.empty:
                continue
            xs = traj_df[["pos_x", "pos_y", "pos_z"]].values
            if len(xs) < 2:
                continue

            # Find first hit in recorded data
            hit_idx = None
            hit_target_idx = None
            for i, pt in enumerate(xs):
                for idx, ht in enumerate(hitbox_targets):
                    if is_point_in_hitbox(pt, ht["box"]):
                        hit_idx = i
                        hit_target_idx = idx
                        break
                if hit_idx is not None:
                    break

            # Keep only trajectories that hit a target in recorded data
            if hit_idx is None:
                continue

            # Truncate at first hit
            xs_bif = xs[:hit_idx + 1]
            if len(xs_bif) < 2:
                continue

            # --- Bifurcation detection ---
            bifurcation_targets = target_centers.copy()
            # Compute landing point for the hit target
            hit_pt = xs[hit_idx]
            box = hitbox_targets[hit_target_idx]["box"]
            c = box["center"]
            n_vec = box["n"]
            v = hit_pt - c
            dist_to_plane = np.dot(v, n_vec)
            landing_point = hit_pt - dist_to_plane * n_vec
            bifurcation_targets[hit_target_idx] = landing_point

            results = find_bifurcations(
                xs_bif, bifurcation_targets,
                required_hold_steps=REQUIRED_HOLD_STEPS,
                window_size=WINDOW_SIZE,
                min_corr_threshold=MIN_CORR_THRESHOLD,
                plateau_hold_steps=PLATEAU_HOLD_STEPS,
                next_start=NEXT_START,
                trajectory_fraction=TRAJECTORY_FRACTION
            )
            n_bif = len(results["bif_indices"])
            if n_bif > 2:
                n_bif = 2   # cap at 2 for grouping

            # --- Initial target ---
            initial_target = initial_max_angle_target_with_landing(
                xs_bif, target_centers, hit_idx, hit_target_idx, hitbox_targets, fraction=0.05
            )

            # --- Update counters ---
            total_valid += 1
            if initial_target == hit_target_idx:
                match_total += 1
                counts[n_bif][1] += 1
            counts[n_bif][0] += 1

# ============================================================
# Report
# ============================================================
print(f"\nOverall: {total_valid} trajectories, matches = {match_total} ({100*match_total/total_valid:.1f}%)")
for n_bif in [0, 1, 2]:
    tot, mat = counts[n_bif]
    if tot > 0:
        print(f"  {n_bif} bifurcation(s): {tot} trajectories, matches = {mat} ({100*mat/tot:.1f}%)")
    else:
        print(f"  {n_bif} bifurcation(s): no trajectories")

Processing 30.10.24 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-

Processing 24.12.24 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-

Processing 26.12.24 ...


...

C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-

Processing 01.10.22 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-

Processing 27.10.22 ...

Overall: 347 trajectories, matches = 149 (42.9%)
  0 bifurcation(s): 49 trajectories, matches = 4 (8.2%)
  1 bifurcation(s): 211 trajectories, matches = 99 (46.9%)
  2 bifurcation(s): 87 trajectories, matches = 46 (52.9%)


C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_5404\1702735674.py:160: RuntimeWarning: All-

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Parameters (same as your multi‑trajectory execution)
# ============================================================
REQUIRED_HOLD_STEPS = 25
WINDOW_SIZE = 25
MIN_CORR_THRESHOLD = 0.7
PLATEAU_HOLD_STEPS = 18
NEXT_START = 0
TRAJECTORY_FRACTION = 0.7          # match multi‑trajectory code

# ============================================================
# Define which trajectories to process (all days except 06.10.22)
# ============================================================
SELECTED_TRAJECTORIES = {
    "30.10.24": {
        "brad": list(range(1, 36)),
        "ketem": list(range(1, 16)),
        "Pitt": list(range(1, 52)),
    },
    "24.12.24": {
        "pitt": list(range(1, 62)),
    },
    "26.12.24": {
        "ketem": list(range(1, 73)),
        "motek": list(range(1, 52)),
        "pitt": list(range(1, 62)),
    },
    "31.12.24": {
        "ketem": list(range(1, 62)),
    },
    "03.11.22": {
        "ketem": list(range(1, 33)),
        "pitt": list(range(1, 48)),
    },
    "01.10.22": {
        "ketem": list(range(1, 1012)),
    },
    "27.10.22": {
        "motek": list(range(1, 38)),
    },
    # "06.10.22" deliberately omitted
}

# ============================================================
# Helper: process a single trajectory and return tagged events
# ============================================================
def process_trajectory(date, bat, traj_no):
    df = pd.read_csv(MASTER_FILES[date])
    df["trajectory_no"] = pd.to_numeric(df["trajectory_no"], errors="coerce")
    mask = (df["bat_name"].astype(str) == bat) & (df["trajectory_no"] == traj_no)
    traj_df = df[mask].dropna(subset=["pos_x", "pos_y", "pos_z"])
    if traj_df.empty:
        return []
    xs = traj_df[["pos_x", "pos_y", "pos_z"]].values
    if len(xs) < 2:
        return []

    # Target centres and hitboxes
    rects = get_rectangles(date)
    target_centers = np.array([rectangle_center(rect) for rect in rects])
    hitbox_targets = [{"rect": rect, "box": get_hitbox_params(rect, 200, 50)} for rect in rects]

    # Check if trajectory ever hits a target (recorded data only)
    hit_idx = None
    hit_target_idx = None
    hit_pt = None
    for i, pt in enumerate(xs):
        for idx, ht in enumerate(hitbox_targets):
            if is_point_in_hitbox(pt, ht["box"]):
                hit_idx = i
                hit_target_idx = idx
                hit_pt = pt.copy()
                break
        if hit_idx is not None:
            break

    # Ignore trajectories that never reach any target
    if hit_idx is None:
        return []

    # Truncate at first hit
    xs_bif = xs[:hit_idx+1]
    if len(xs_bif) < 2:
        return []

    # Bifurcation targets with landing point substitution
    bifurcation_targets = target_centers.copy()
    if hit_target_idx is not None and hit_pt is not None:
        box = hitbox_targets[hit_target_idx]["box"]
        c = box["center"]
        n = box["n"]
        v = hit_pt - c
        dist_to_plane = np.dot(v, n)
        landing_point = hit_pt - dist_to_plane * n
        bifurcation_targets[hit_target_idx] = landing_point

    # Run bifurcation detection (with trajectory_fraction and updated role assignment)
    results = find_bifurcations(
        xs_bif, bifurcation_targets,
        required_hold_steps=REQUIRED_HOLD_STEPS,
        window_size=WINDOW_SIZE,
        min_corr_threshold=MIN_CORR_THRESHOLD,
        plateau_hold_steps=PLATEAU_HOLD_STEPS,
        next_start=NEXT_START,
        trajectory_fraction=TRAJECTORY_FRACTION
    )

    return results.get("bifurcations", [])

# ============================================================
# Collect data over all selected days
# ============================================================
rows = []
for date, bat_traj_dict in SELECTED_TRAJECTORIES.items():
    print(f"Processing {date} ...")
    for bat, traj_list in bat_traj_dict.items():
        for traj_no in traj_list:
            bif_events = process_trajectory(date, bat, traj_no)
            n_bif = len(bif_events)
            for ev in bif_events:
                rows.append({
                    "date": date,
                    "bat_name": bat,
                    "trajectory_no": traj_no,
                    "n_bif": n_bif,
                    "role": ev["role"],               # "first" or "second"
                    "typology": ev["typology"],       # "edge-pair" or "edge-center"
                    "critical_angle_deg": ev["critical_angle_deg"]
                })

df_all = pd.DataFrame(rows)
print(f"Total bifurcation events collected: {len(df_all)}")

# ============================================================
# Helper function to add mean line and text (text size now set globally)
# ============================================================
def add_mean(ax, data, color='black'):
    if len(data) > 0:
        mean_val = np.mean(data)
        ax.axvline(mean_val, color=color, linestyle='--', linewidth=3)
        ax.text(0.95, 0.95, f'Mean = {mean_val:.1f}°',
                transform=ax.transAxes, ha='right', va='top',
                fontsize=plt.rcParams['font.size'],
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

# ============================================================
# Global text size boost (3× default)
# ============================================================
plt.rcParams.update({
    'font.size': 24,          # base font size
    'axes.titlesize': 28,
    'axes.labelsize': 26,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
    'figure.titlesize': 30
})
sns.set_style("whitegrid")

# ============================================================
# Combined figures (only pooled data, no per‑day)
# ============================================================

# --- Figure 1: trajectories with exactly 2 bifurcations ---
two_bif = df_all[df_all["n_bif"] == 2]
if not two_bif.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    edge_pair = two_bif[two_bif["typology"] == "edge-pair"]["critical_angle_deg"].dropna()
    edge_center = two_bif[two_bif["typology"] == "edge-center"]["critical_angle_deg"].dropna()
    if len(edge_pair) > 0:
        sns.histplot(edge_pair, bins=15, kde=True, color="steelblue", ax=ax1)
        ax1.set_title(f"Edge-pair (n={len(edge_pair)})")
        ax1.set_xlabel("Critical angle (°)")
        ax1.set_ylabel("Count")
        add_mean(ax1, edge_pair)
    else:
        ax1.text(0.5, 0.5, "No data", transform=ax1.transAxes, ha='center')
    if len(edge_center) > 0:
        sns.histplot(edge_center, bins=15, kde=True, color="salmon", ax=ax2)
        ax2.set_title(f"Edge-center (n={len(edge_center)})")
        ax2.set_xlabel("Critical angle (°)")
        ax2.set_ylabel("Count")
        add_mean(ax2, edge_center)
    else:
        ax2.text(0.5, 0.5, "No data", transform=ax2.transAxes, ha='center')
    fig.suptitle("Trajectories with 2 bifurcations", fontsize=plt.rcParams['figure.titlesize'])
    plt.tight_layout()
    plt.show()
else:
    print("No trajectories with 2 bifurcations found.")

# --- Figure 2: trajectories with exactly 1 bifurcation ---
one_bif = df_all[df_all["n_bif"] == 1]
if not one_bif.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    edge_pair = one_bif[one_bif["typology"] == "edge-pair"]["critical_angle_deg"].dropna()
    edge_center = one_bif[one_bif["typology"] == "edge-center"]["critical_angle_deg"].dropna()
    if len(edge_pair) > 0:
        sns.histplot(edge_pair, bins=15, kde=True, color="steelblue", ax=ax1)
        ax1.set_title(f"Edge-pair (n={len(edge_pair)})")
        ax1.set_xlabel("Critical angle (°)")
        ax1.set_ylabel("Count")
        add_mean(ax1, edge_pair)
    else:
        ax1.text(0.5, 0.5, "No data", transform=ax1.transAxes, ha='center')
    if len(edge_center) > 0:
        sns.histplot(edge_center, bins=15, kde=True, color="salmon", ax=ax2)
        ax2.set_title(f"Edge-center (n={len(edge_center)})")
        ax2.set_xlabel("Critical angle (°)")
        ax2.set_ylabel("Count")
        add_mean(ax2, edge_center)
    else:
        ax2.text(0.5, 0.5, "No data", transform=ax2.transAxes, ha='center')
    fig.suptitle("Trajectories with 1 bifurcation", fontsize=plt.rcParams['figure.titlesize'])
    plt.tight_layout()
    plt.show()
else:
    print("No trajectories with 1 bifurcation found.")

# --- Figure 3: all bifurcations pooled together ---
if not df_all.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    edge_pair = df_all[df_all["typology"] == "edge-pair"]["critical_angle_deg"].dropna()
    edge_center = df_all[df_all["typology"] == "edge-center"]["critical_angle_deg"].dropna()
    if len(edge_pair) > 0:
        sns.histplot(edge_pair, bins=15, kde=True, color="steelblue", ax=ax1)
        ax1.set_title(f"All edge-pair (n={len(edge_pair)})")
        ax1.set_xlabel("Critical angle (°)")
        ax1.set_ylabel("Count")
        add_mean(ax1, edge_pair)
    else:
        ax1.text(0.5, 0.5, "No data", transform=ax1.transAxes, ha='center')
    if len(edge_center) > 0:
        sns.histplot(edge_center, bins=15, kde=True, color="salmon", ax=ax2)
        ax2.set_title(f"All edge-center (n={len(edge_center)})")
        ax2.set_xlabel("Critical angle (°)")
        ax2.set_ylabel("Count")
        add_mean(ax2, edge_center)
    else:
        ax2.text(0.5, 0.5, "No data", transform=ax2.transAxes, ha='center')
    fig.suptitle("All bifurcations combined", fontsize=plt.rcParams['figure.titlesize'])
    plt.tight_layout()
    plt.show()
else:
    print("No bifurcation events collected at all.")

# ============================================================
# Additional summary: count of first/second bifurcations by typology
# ============================================================
if not df_all.empty:
    summary = df_all.groupby(["role", "typology"]).size().unstack(fill_value=0)
    print("\nCount of bifurcations by role and typology:")
    print(summary)
    
    # Create a bar chart
    fig, ax = plt.subplots(figsize=(10, 8))
    summary.plot(kind='bar', stacked=False, ax=ax, color=["steelblue", "salmon"], edgecolor='black')
    ax.set_title("Number of bifurcations by chronological order and typology")
    ax.set_xlabel("Chronological role")
    ax.set_ylabel("Count")
    ax.legend(title="Typology")
    plt.tight_layout()
    plt.show()

Processing 30.10.24 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarnin

Processing 24.12.24 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarnin

Processing 26.12.24 ...


...

Processing 03.11.22 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarnin

Processing 01.10.22 ...


C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarning: All-NaN slice encountered
  min_corr = np.nanmin(pair_corr, axis=1)
C:\Users\iswav\AppData\Local\Temp\ipykernel_15272\1702735674.py:160: RuntimeWarnin

Processing 27.10.22 ...
Total bifurcation events collected: 378

Count of bifurcations by role and typology:
typology  edge-center  edge-pair
role                            
first             216         76
second             72         14


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

# =====================================================
# 3D GEOMETRY HELPERS (same as before)
# =====================================================
def get_hitbox_params(rect, depth=300, side_padding=0):
    p0, p1, p2, p3 = rect[0], rect[1], rect[2], rect[3]
    u = p1 - p0
    v = p3 - p0
    L_u = np.linalg.norm(u) + (side_padding * 2)
    L_v = np.linalg.norm(v) + (side_padding * 2)
    u_hat = u / np.linalg.norm(u)
    v_hat = v / np.linalg.norm(v)
    n_hat = np.cross(u_hat, v_hat)
    center = (p0 + p2) / 2.0
    return {"center": center, "u": u_hat, "L_u": L_u,
            "v": v_hat, "L_v": L_v, "n": n_hat, "L_n": depth}

def is_point_in_hitbox(pt, box):
    v = pt - box["center"]
    if abs(np.dot(v, box["u"])) > box["L_u"] / 2.0: return False
    if abs(np.dot(v, box["v"])) > box["L_v"] / 2.0: return False
    if abs(np.dot(v, box["n"])) > box["L_n"] / 2.0: return False
    return True

RECT_COLORS = ["black", "magenta", "cyan", "orange", "purple"]

# =====================================================
# EXTENDED STATISTICS FUNCTION
# =====================================================
def compute_target_statistics(
    hitbox_depth=200,
    hitbox_padding=50,
    extrapolation_step=10,
    max_extrapolation=200,
):
    """
    Returns
    -------
    per_date_stats : dict
        date -> {
            "total_trajectories": int,
            "num_targets": int,
            "hits": dict {target_idx: count},
            "start_stats": dict {target_idx: {
                "mean": ndarray (3,),
                "std_per_axis": ndarray (3,),   # σ_x, σ_y, σ_z
                "scatter": float                # mean Euclidean distance from mean
            } or None if no hits},
            "pairwise_distances": dict {(i,j): float or None}
        }
    overall_probability : dict
        target_idx -> probability (float)
    """
    per_date_stats = {}
    overall_hits = defaultdict(int)
    overall_total_traj = 0
    exclude_overall = {"06.10.22"}   # date with only 2 targets

    for date, filepath in MASTER_FILES.items():
        print(f"\nProcessing {date} ...")
        df = pd.read_csv(filepath)
        df["trajectory_no"] = pd.to_numeric(df["trajectory_no"], errors="coerce")

        # Unique trajectories
        unique_trajs = df[["bat_name", "trajectory_no"]].drop_duplicates()
        total = len(unique_trajs)

        # Targets for this date
        rects = get_rectangles(date)
        num_targets = len(rects)

        # Build hitboxes
        hitbox_targets = [
            {
                "name": f"Rectangle {i+1}",
                "rect": rect,
                "color": RECT_COLORS[i % len(RECT_COLORS)],
                "box": get_hitbox_params(rect, hitbox_depth, hitbox_padding)
            }
            for i, rect in enumerate(rects)
        ]

        hits = {i: 0 for i in range(num_targets)}
        start_positions = {i: [] for i in range(num_targets)}

        # Classify each trajectory
        for (bat, traj) in unique_trajs.itertuples(index=False):
            traj_df = df[(df["bat_name"]==bat) & (df["trajectory_no"]==traj)].copy()
            traj_df = traj_df.dropna(subset=["pos_x", "pos_y", "pos_z"])
            if traj_df.empty:
                continue
            xs = traj_df[["pos_x", "pos_y", "pos_z"]].values
            if len(xs) < 2:
                continue

            start_pt = xs[0]

            # Landing classification
            hit_target_idx = None
            for pt in xs:
                for idx, ht in enumerate(hitbox_targets):
                    if is_point_in_hitbox(pt, ht["box"]):
                        hit_target_idx = idx
                        break
                if hit_target_idx is not None:
                    break

            # Extrapolate if needed
            if hit_target_idx is None:
                tail_len = max(2, int(len(xs) * 0.1))
                velocity = xs[-1] - xs[-tail_len]
                speed = np.linalg.norm(velocity)
                if speed > 0:
                    direction = velocity / speed
                    curr_pt = xs[-1].copy()
                    for _ in range(max_extrapolation):
                        curr_pt = curr_pt + direction * extrapolation_step
                        for idx, ht in enumerate(hitbox_targets):
                            if is_point_in_hitbox(curr_pt, ht["box"]):
                                hit_target_idx = idx
                                break
                        if hit_target_idx is not None:
                            break

            if hit_target_idx is not None:
                hits[hit_target_idx] += 1
                start_positions[hit_target_idx].append(start_pt)

        # Compute start statistics per target
        avg_start = {}
        start_stats = {}
        for i in range(num_targets):
            pts = start_positions[i]
            if pts:
                arr = np.array(pts)                     # shape (n, 3)
                mean = arr.mean(axis=0)
                std_per_axis = arr.std(axis=0)          # shape (3,)
                dists = np.linalg.norm(arr - mean, axis=1)
                scatter = dists.mean()
                avg_start[i] = mean
                start_stats[i] = {
                    "mean": mean,
                    "std_per_axis": std_per_axis,
                    "scatter": scatter
                }
            else:
                avg_start[i] = None
                start_stats[i] = None

        # Pairwise distances between mean start points
        pairwise_dist = {}
        for i in range(num_targets):
            for j in range(i+1, num_targets):
                if avg_start[i] is not None and avg_start[j] is not None:
                    d = np.linalg.norm(avg_start[i] - avg_start[j])
                    pairwise_dist[(i, j)] = d
                else:
                    pairwise_dist[(i, j)] = None

        per_date_stats[date] = {
            "total_trajectories": total,
            "num_targets": num_targets,
            "hits": hits,
            "start_stats": start_stats,
            "pairwise_distances": pairwise_dist
        }

        # Accumulate overall (exclude dates with not 3 targets)
        if date not in exclude_overall:
            overall_total_traj += total
            for i, cnt in hits.items():
                overall_hits[i] += cnt

    # Overall probability (target indices up to max seen)
    max_targets = max(info["num_targets"] for info in per_date_stats.values())
    overall_probability = {}
    for i in range(max_targets):
        overall_probability[i] = overall_hits.get(i, 0) / overall_total_traj if overall_total_traj > 0 else 0.0

    return per_date_stats, overall_probability

In [ ]:
# =====================================================
# RUN THE EXTENDED STATISTICS
# =====================================================
per_date_stats, overall_prob = compute_target_statistics()

# ---- Per‑date summary ----
for date, info in per_date_stats.items():
    print(f"\n{'='*60}")
    print(f"Date: {date}")
    print(f"Total trajectories: {info['total_trajectories']}")
    print(f"Number of targets: {info['num_targets']}")
    for t_idx in range(info['num_targets']):
        hits = info['hits'][t_idx]
        stats = info['start_stats'][t_idx]
        if stats is not None:
            mean = stats["mean"]
            std_xyz = stats["std_per_axis"]
            scatter = stats["scatter"]
            print(f"  Target {t_idx}: {hits} hits")
            print(f"    Mean start = ({mean[0]:.1f}, {mean[1]:.1f}, {mean[2]:.1f})")
            print(f"    Std (x,y,z) = ({std_xyz[0]:.1f}, {std_xyz[1]:.1f}, {std_xyz[2]:.1f})")
            print(f"    Mean distance from mean start = {scatter:.1f}")
        else:
            print(f"  Target {t_idx}: 0 hits")
    
    # Print pairwise distances
    pairwise = info["pairwise_distances"]
    if pairwise:
        print("  Pairwise distances between mean start positions:")
        for (i, j), d in pairwise.items():
            if d is not None:
                print(f"    Target {i} ↔ Target {j}: {d:.1f}")
            else:
                print(f"    Target {i} ↔ Target {j}: N/A (one target had no hits)")

# ---- Overall probability (days with 3 targets, excl. 06.10.22) ----
print(f"\n{'='*60}")
print("Overall hit probability (days with 3 targets, excl. 06.10.22):")
for t_idx, prob in overall_prob.items():
    print(f"  Target {t_idx}: {prob:.3f} ({prob*100:.1f}%)")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================
# 1. PER‑DAY BAR PLOTS (hits per target)
# =====================================================
for date, info in per_date_stats.items():
    targets = list(info["hits"].keys())
    counts = [info["hits"][t] for t in targets]
    
    fig = go.Figure(data=[
        go.Bar(
            x=[f"Target {t}" for t in targets],
            y=counts,
            marker_color=[RECT_COLORS[t % len(RECT_COLORS)] for t in targets],
            text=counts,
            textposition='outside'
        )
    ])
    fig.update_layout(
        title=f"Target hit distribution – {date}",
        xaxis_title="Target",
        yaxis_title="Number of hits",
        yaxis=dict(range=[0, max(counts)*1.15]),
        template="simple_white",
        width=600,
        height=450
    )
    fig.show()

# =====================================================
# 2. COMBINED BAR PLOT (overall probability, 3‑target days only)
# =====================================================
target_indices = sorted(overall_prob.keys())
probs = [overall_prob[t] * 100 for t in target_indices]  # percentages

fig_combined = go.Figure(data=[
    go.Bar(
        x=[f"Target {t}" for t in target_indices],
        y=probs,
        marker_color=[RECT_COLORS[t % len(RECT_COLORS)] for t in target_indices],
        text=[f"{p:.1f}%" for p in probs],
        textposition='outside'
    )
])
fig_combined.update_layout(
    title="Overall hit probability (all 3‑target days, excl. 06.10.22)",
    xaxis_title="Target",
    yaxis_title="Probability (%)",
    yaxis=dict(range=[0, max(probs)*1.15]),
    template="simple_white",
    width=600,
    height=450
)
fig_combined.show()